# Does AI materials discovery lean on critical raw materials?

An audit of Google DeepMind's GNoME dataset (~554,000 computed stable materials): are its materials made of
scarce, supply-risky elements more often than known materials, and why?

**Author:** Mohammad Haris Mushtaq · **Code licence:** MIT · **Results licence:** CC BY 4.0

| Level | What it checks | Sections |
|---|---|---|
| 1 | The main result is solid: two critical lists (EU, US), atom and mass share, duplicates, fair matching | 6–8 |
| 2 | *Why* it happens: chemistry mix, inherited bias, near-copies, stability margins, r²SCAN | 9–14 |
| 3 | Does it matter in practice: application shortlists, country exposure, lower-risk alternatives, other databases | 15–18 |
| 4 | Own calculations: re-check a sample with a universal ML potential (optional, GPU; not run for the paper) | 19 |
| — | Null calibration, extra checks for the limits, headline table | 20–22 |

**Honesty rules.** H1–H3 were fixed before the first real run. Everything else is *exploratory*.
One analysis was changed after seeing results: the first lower-risk-alternatives design (section 17, first cell)
counted almost identical candidates and gave an implausible 100% / zero-loss result, so it was redesigned
(section 17, second cell). Both versions are kept and saved.

**How to run.**
1. Open in Google Colab. No GPU is needed (except the optional Level 4).
2. Run everything once with `DEMO_MODE = True` (fake data, a few minutes) to check the pipeline.
3. Set `DEMO_MODE = False`, add your free Materials Project API key (Colab *Secrets* → `MP_API_KEY`), and run all.
   The first real run downloads ~2 GB and takes a few hours; downloads and the main table are cached on Google Drive,
   so later runs are much faster.
4. All tables and figures are written to `results/` and downloaded as a zip at the end.

In [ ]:
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymatgen", "mp-api",
                    "scikit-learn", "pyarrow", "fsspec", "aiohttp", "ase"], check=True)
print("Colab:", IN_COLAB)

## 1. Analysis code
Three files: the original tests (v1), the extended analyses (v2), country exposure and alternatives (v3).

In [ ]:
%%writefile crm_audit.py
"""
crm_audit.py — Does AI materials discovery over-produce critical-element compounds,
and are those "discoveries" artifacts of known errors in the training data?

Three tests:
  A. Criticality: are AI-proposed (GNoME) compounds more supply-risky than known
     compounds, after matching on number of elements?
  B. Link: are the elements that AI over-produces the same elements whose energies
     change most between GGA(+U) and the more accurate r2SCAN method?
  C. Survival: do AI-proposed compounds with those elements lose stability under
     r2SCAN more often than comparable known compounds?
Plus an optional "artifact risk" classifier, evaluated with grouped cross-validation.
"""
from __future__ import annotations

import os
import math
import numpy as np
import pandas as pd
from scipy import stats

# --------------------------------------------------------------------------------------
# Element groups (edit here; every analysis reads from these lists)
# --------------------------------------------------------------------------------------
LANTHANIDES = ["La", "Ce", "Pr", "Nd", "Pm", "Sm", "Eu", "Gd", "Tb", "Dy",
               "Ho", "Er", "Tm", "Yb", "Lu"]
REE = LANTHANIDES + ["Sc", "Y"]

# Lanthanides whose 4f electrons are frozen in the core ("_3"/"_2" PAW potentials) in
# Materials Project calculations. Verified in pymatgen's MP input sets: MPRelaxSet.yaml
# (GGA) and MPSCANRelaxSet.yaml (r2SCAN) both use Pr_3, Nd_3, Sm_3, Tb_3, Dy_3, Ho_3,
# Er_3, Tm_3, Lu_3 and Yb_2/Yb_3; La, Ce, Eu and Gd use 4f-in-valence potentials.
# Pm has no MP potential choice listed; it is kept with its neighbours.
F_IN_CORE = ["Pr", "Nd", "Pm", "Sm", "Tb", "Dy", "Ho", "Er", "Tm", "Yb", "Lu"]

# Elements that get a Hubbard U correction in MP oxides/fluorides only (mixing scheme).
U_ELEMENTS = ["V", "Cr", "Mn", "Fe", "Co", "Ni", "Mo", "W"]

# Radioactive elements with no stable isotope (not usable as real materials at scale).
RADIOACTIVE = ["Tc", "Pm", "Po", "At", "Rn", "Fr", "Ra", "Ac", "Th", "Pa", "U",
               "Np", "Pu", "Am", "Cm", "Bk", "Cf", "Es", "Fm", "Md", "No", "Lr"]

# EU 2023 Critical Raw Materials list, mapped to elements.
# Clear one-to-one cases only. Raw materials that do not map cleanly onto an element
# in a compound (bauxite->Al, fluorspar->F, natural graphite->C, silicon metal->Si,
# coking coal, feldspar, phosphate rock, helium) are kept apart in CRM_AMBIGUOUS so
# you can test both versions. VERIFY against the official EU list before publishing.
CRM_EU2023 = (["Sb", "As", "Ba", "Be", "Bi", "B", "Co", "Cu", "Ga", "Ge", "Hf",
               "Li", "Mg", "Mn", "Ni", "Nb", "P", "Sr", "Ta", "Ti", "W", "V",
               "Ru", "Rh", "Pd", "Os", "Ir", "Pt"] + REE)
CRM_AMBIGUOUS = ["Al", "F", "C", "Si", "He"]

THERMO_GGA = "GGA_GGA+U"
THERMO_MIX = "GGA_GGA+U_R2SCAN"
THERMO_R2 = "R2SCAN"


# --------------------------------------------------------------------------------------
# 1. Data loading (Materials Project via mp-api). Everything is cached to disk.
# --------------------------------------------------------------------------------------
SUMMARY_FIELDS = ["material_id", "formula_pretty", "composition_reduced", "elements",
                  "nelements", "nsites", "theoretical", "energy_above_hull",
                  "is_stable", "formation_energy_per_atom"]
THERMO_FIELDS = ["material_id", "thermo_type", "energy_above_hull", "is_stable",
                 "formation_energy_per_atom"]


def _cached(path, builder):
    """Load a parquet file if it exists, otherwise build it and save it."""
    if path and os.path.exists(path):
        return pd.read_parquet(path)
    df = builder()
    if path:
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        df.to_parquet(path, index=False)
    return df


def _is_el(sym):
    from pymatgen.core.periodic_table import Element
    return Element.is_valid_symbol(sym)


def _summary_docs_to_df(docs):
    rows = []
    for d in docs:
        d = d if isinstance(d, dict) else d.model_dump()
        comp = d.get("composition_reduced")
        if hasattr(comp, "as_dict"):
            comp = comp.as_dict()
        comp = {str(k): float(v) for k, v in (comp or {}).items() if _is_el(str(k))}
        rows.append({
            "material_id": str(d["material_id"]),
            "formula": d.get("formula_pretty"),
            "composition": comp,
            "elements": sorted(str(getattr(e, "symbol", e)) for e in (d.get("elements") or [])),
            "nelements": d.get("nelements"),
            "nsites": d.get("nsites"),
            "theoretical": d.get("theoretical"),
            "e_hull_default": d.get("energy_above_hull"),
            "is_stable_default": d.get("is_stable"),
            "ef_default": d.get("formation_energy_per_atom"),
        })
    df = pd.DataFrame(rows)
    # parquet cannot store dicts with varying keys well -> store composition as JSON-like str
    df["composition"] = df["composition"].apply(lambda c: ";".join(f"{k}:{v}" for k, v in c.items()))
    df["elements"] = df["elements"].apply(lambda e: ",".join(e))
    return df


def parse_elements(s):
    return s.split(",") if isinstance(s, str) and s else []


def parse_composition(s):
    out = {}
    if isinstance(s, str) and s:
        for part in s.split(";"):
            k, v = part.split(":")
            out[k] = float(v)
    return out


def fetch_summary(api_key, include_gnome, cache_path=None, chunk_size=1000):
    def build():
        from mp_api.client import MPRester
        with MPRester(api_key, use_document_model=False, mute_progress_bars=False) as mpr:
            docs = mpr.materials.summary.search(
                fields=SUMMARY_FIELDS, deprecated=False,
                include_gnome=include_gnome, chunk_size=chunk_size)
        return _summary_docs_to_df(docs)
    return _cached(cache_path, build)


def fetch_thermo(api_key, thermo_type, cache_path=None, chunk_size=1000):
    def build():
        from mp_api.client import MPRester
        with MPRester(api_key, use_document_model=False, mute_progress_bars=False) as mpr:
            docs = mpr.materials.thermo.search(
                thermo_types=[thermo_type], fields=THERMO_FIELDS, chunk_size=chunk_size)
        rows = []
        for d in docs:
            d = d if isinstance(d, dict) else d.model_dump()
            rows.append({
                "material_id": str(d["material_id"]),
                "thermo_type": str(getattr(d.get("thermo_type"), "value", d.get("thermo_type"))),
                "e_hull": d.get("energy_above_hull"),
                "is_stable": d.get("is_stable"),
                "ef": d.get("formation_energy_per_atom"),
            })
        return pd.DataFrame(rows)
    return _cached(cache_path, build)


def label_groups(df_with_gnome, df_without_gnome):
    """GNoME = present only when GNoME is included. Known = MP non-theoretical
    (has an experimental ICSD record). Theoretical = other computed proposals."""
    ids_without = set(df_without_gnome["material_id"])
    df = df_with_gnome.copy()
    df["group"] = np.where(~df["material_id"].isin(ids_without), "GNoME",
                   np.where(df["theoretical"].astype(bool), "MP_theoretical", "MP_known"))
    return df


GNOME_CSV_URL = ("https://storage.googleapis.com/gdm_materials_discovery/"
                 "gnome_data/stable_materials_summary.csv")


def load_gnome_csv(url=GNOME_CSV_URL, cache_path=None):
    """DeepMind's own GNoME summary table (PBE energies from GNoME's hull).
    Column names are detected loosely because they may change between releases."""
    def build():
        return pd.read_csv(url)
    raw = _cached(cache_path, build)
    cols = {c.lower().replace(" ", "_"): c for c in raw.columns}
    def pick(*cands):
        for c in cands:
            if c in cols:
                return cols[c]
        raise KeyError(f"None of {cands} in columns: {list(raw.columns)}")
    out = pd.DataFrame({
        "gnome_id": raw[pick("materialid", "material_id")],
        "reduced_formula": raw[pick("reduced_formula", "reduced_formula_")],
        "nsites": raw[pick("nsites", "n_sites")],
        "gnome_e_decomp": raw[pick("decomposition_energy_per_atom")],
    })
    return out


def match_gnome_csv(df_meta, gnome_csv):
    """Attach GNoME's own decomposition energy to MP GNoME entries by matching
    reduced formula + number of sites. Ambiguous matches (several rows) are dropped."""
    from pymatgen.core import Composition
    g = df_meta[df_meta["group"] == "GNoME"].copy()
    g["reduced_formula"] = g["formula"].apply(lambda f: Composition(f).reduced_formula)
    c = gnome_csv.copy()
    c["reduced_formula"] = c["reduced_formula"].apply(lambda f: Composition(f).reduced_formula)
    c = c.drop_duplicates(["reduced_formula", "nsites"], keep=False)
    m = g.merge(c, on=["reduced_formula", "nsites"], how="inner")
    return m[["material_id", "gnome_e_decomp"]]


def coverage_table(df_meta, thermo_by_type):
    """How many compounds in each group have each thermo type. Read this first:
    it tells you which comparisons are actually possible."""
    rows = []
    for g, d in df_meta.groupby("group"):
        row = {"group": g, "n": len(d)}
        ids = set(d["material_id"])
        for t, th in thermo_by_type.items():
            row[t] = int(th["material_id"].isin(ids).sum())
        rows.append(row)
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------------------
# 2. Compound-level criticality metrics
# --------------------------------------------------------------------------------------
def element_hhi_table():
    """Element HHI (production, reserves) from pymatgen (Gaultois et al., 2013).
    NOTE: these values reflect ~2011 USGS data; consider updating with current USGS."""
    from pymatgen.analysis.hhi import HHIModel
    h = HHIModel()
    return {el: (float(p), float(r)) for el, (p, r) in h.symbol_hhip_hhir.items()}


def add_compound_metrics(df, hhi=None, crm=None):
    """Adds: hhi_p / hhi_r (atom-fraction weighted), hhi_p_max, crm_frac, flags."""
    hhi = hhi or element_hhi_table()
    crm = set(crm or CRM_EU2023)
    f_core, ree, uel, rad = set(F_IN_CORE), set(REE), set(U_ELEMENTS), set(RADIOACTIVE)
    out = {k: [] for k in ["hhi_p", "hhi_r", "hhi_p_max", "crm_frac", "hhi_missing"]}
    for s in df["composition"].values:
        comp = parse_composition(s)
        tot = sum(comp.values()) or 1.0
        hp = hr = known = crm_amt = 0.0
        hmax = 0.0
        missing = False
        for el, amt in comp.items():
            frac = amt / tot
            if el in hhi:
                hp += frac * hhi[el][0]
                hr += frac * hhi[el][1]
                known += frac
                hmax = max(hmax, hhi[el][0])
            else:
                missing = True
            if el in crm:
                crm_amt += frac
        out["hhi_p"].append(hp / known if known else np.nan)
        out["hhi_r"].append(hr / known if known else np.nan)
        out["hhi_p_max"].append(hmax if known else np.nan)
        out["crm_frac"].append(crm_amt)
        out["hhi_missing"].append(missing)
    df = df.copy()
    for k, v in out.items():
        df[k] = v
    els = df["elements"].apply(lambda s: set(parse_elements(s)))
    df["has_ree"] = els.apply(lambda e: bool(e & ree))
    df["has_f_in_core"] = els.apply(lambda e: bool(e & f_core))
    df["has_u_element"] = els.apply(lambda e: bool(e & uel))
    df["has_radioactive"] = els.apply(lambda e: bool(e & rad))
    df["chemsys"] = df["elements"].apply(lambda s: "-".join(sorted(parse_elements(s))))
    return df


# --------------------------------------------------------------------------------------
# 3. Test A — criticality and element enrichment, matched on number of elements
# --------------------------------------------------------------------------------------
def _strata_weights(target, ref, strata_col):
    """Weights that make `ref` have the same distribution of `strata_col` as `target`
    (direct standardisation). Strata absent from ref are dropped from target."""
    t = target[strata_col].value_counts(normalize=True)
    r = ref[strata_col].value_counts(normalize=True)
    common = t.index.intersection(r.index)
    t = t[common] / t[common].sum()
    w = ref[strata_col].map(lambda s: t.get(s, 0.0) / r[s] if s in common else 0.0)
    return w.values, set(common)


def compare_metric(df, metric, target="GNoME", ref="MP_known", strata_col="nelements",
                   n_boot=500, seed=0):
    """Difference in mean `metric` between target and ref, with ref reweighted to
    the target's nelements mix. Bootstrap CI (resampling compounds in both groups)."""
    rng = np.random.default_rng(seed)
    T = df[df["group"] == target].dropna(subset=[metric])
    R = df[df["group"] == ref].dropna(subset=[metric])
    w, common = _strata_weights(T, R, strata_col)
    T = T[T[strata_col].isin(common)]
    tv, rv = T[metric].astype(float).values, R[metric].astype(float).values
    point_t, point_r = tv.mean(), np.average(rv, weights=w)
    boots = []
    for _ in range(n_boot):
        ti = rng.integers(0, len(tv), len(tv))
        ri = rng.integers(0, len(rv), len(rv))
        wr = w[ri]
        if wr.sum() == 0:
            continue
        boots.append(tv[ti].mean() - np.average(rv[ri], weights=wr))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {"metric": metric, "target": target, "ref": ref,
            "mean_target": point_t, "mean_ref_matched": point_r,
            "mean_ref_raw": rv.mean(), "diff": point_t - point_r,
            "ci_low": lo, "ci_high": hi, "n_target": len(tv), "n_ref": len(rv)}


def element_indicator(df, elements=None):
    lists = df["elements"].apply(parse_elements)
    elements = elements or sorted({e for l in lists for e in l})
    idx = {e: i for i, e in enumerate(elements)}
    M = np.zeros((len(df), len(elements)), dtype=bool)
    for r, l in enumerate(lists):
        for e in l:
            if e in idx:
                M[r, idx[e]] = True
    return M, elements


def element_enrichment(df, target="GNoME", ref="MP_known", strata_col="nelements",
                       n_boot=200, seed=0, min_count=20):
    """For each element: share of target compounds containing it vs the
    nelements-matched share in ref. Returns log2 ratio with bootstrap CI."""
    rng = np.random.default_rng(seed)
    T = df[df["group"] == target]
    R = df[df["group"] == ref]
    w, common = _strata_weights(T, R, strata_col)
    T = T[T[strata_col].isin(common)]
    MT, els = element_indicator(T)
    MR, _ = element_indicator(R, els)
    eps = 0.5  # pseudo-count to keep log finite
    def ratio(mt, mr, wr):
        pt = (mt.sum(0) + eps) / (len(mt) + 2 * eps)
        pr = ((mr * wr[:, None]).sum(0) + eps * wr.mean()) / (wr.sum() + 2 * eps * wr.mean())
        return np.log2(pt / pr), pt, pr
    lr, pt, pr = ratio(MT, MR, w)
    boots = np.empty((n_boot, len(els)))
    for b in range(n_boot):
        ti = rng.integers(0, len(MT), len(MT))
        ri = rng.integers(0, len(MR), len(MR))
        boots[b] = ratio(MT[ti], MR[ri], w[ri])[0]
    lo, hi = np.percentile(boots, [2.5, 97.5], axis=0)
    res = pd.DataFrame({"element": els, "share_target": pt, "share_ref_matched": pr,
                        "log2_enrichment": lr, "ci_low": lo, "ci_high": hi,
                        "count_target": MT.sum(0), "count_ref": MR.sum(0)})
    res = res[(res["count_target"] + res["count_ref"]) >= min_count]
    return res.sort_values("log2_enrichment", ascending=False).reset_index(drop=True)


# --------------------------------------------------------------------------------------
# 4. Test B — link enrichment to a per-element "data reliability" signal
# --------------------------------------------------------------------------------------
def element_functional_sensitivity(df_meta, thermo_a, thermo_b, groups=("MP_known",),
                                   value="ef", min_count=10):
    """Per element: median |value_b - value_a| over compounds (in `groups`) that have
    both thermo types. Large = energies for this element change a lot between
    GGA(+U) and r2SCAN = less trustworthy training labels."""
    a = thermo_a[["material_id", value]].rename(columns={value: "a"})
    b = thermo_b[["material_id", value]].rename(columns={value: "b"})
    m = df_meta[df_meta["group"].isin(groups)][["material_id", "elements"]]
    j = m.merge(a, on="material_id").merge(b, on="material_id").dropna(subset=["a", "b"])
    j["absdiff"] = (j["b"] - j["a"]).abs()
    rows = {}
    for els, d in zip(j["elements"], j["absdiff"]):
        for e in parse_elements(els):
            rows.setdefault(e, []).append(d)
    out = pd.DataFrame([{"element": e, "sensitivity": float(np.median(v)), "n_pairs": len(v)}
                        for e, v in rows.items()])
    return out[out["n_pairs"] >= min_count].reset_index(drop=True)


def _rank_residual(y, x):
    ry, rx = stats.rankdata(y), stats.rankdata(x)
    beta = np.polyfit(rx, ry, 1)
    return ry - np.polyval(beta, rx)


def link_test(enrich, sens, data_counts=None, n_perm=10000, seed=0):
    """Spearman correlation between element enrichment and element sensitivity.
    If data_counts (element -> number of known compounds) is given, also a partial
    correlation that removes the effect of how much data each element has.
    Permutation p-values (two-sided)."""
    rng = np.random.default_rng(seed)
    j = enrich.merge(sens, on="element")
    if data_counts is not None:
        j["log_count"] = np.log10(j["element"].map(data_counts).fillna(1).clip(lower=1))
    x, y = j["log2_enrichment"].values, j["sensitivity"].values
    rho = stats.spearmanr(x, y).statistic
    perm = np.array([stats.spearmanr(x, rng.permutation(y)).statistic for _ in range(n_perm)])
    out = {"n_elements": len(j), "spearman_rho": rho,
           "p_perm": (np.sum(np.abs(perm) >= abs(rho)) + 1) / (n_perm + 1)}
    if data_counts is not None:
        rx, ry = _rank_residual(x, j["log_count"]), _rank_residual(y, j["log_count"])
        prho = stats.pearsonr(rx, ry).statistic
        pperm = np.array([stats.pearsonr(rx, rng.permutation(ry)).statistic for _ in range(n_perm)])
        out.update({"partial_rho_given_datacount": prho,
                    "p_perm_partial": (np.sum(np.abs(pperm) >= abs(prho)) + 1) / (n_perm + 1)})
    return out, j


def leave_one_out_link(enrich, sens, data_counts=None):
    """Robustness: drop each element once and recompute rho. A result driven by
    one element is fragile."""
    rows = []
    for e in enrich.merge(sens, on="element")["element"]:
        r, _ = link_test(enrich[enrich.element != e], sens[sens.element != e],
                         data_counts, n_perm=1)
        rows.append({"dropped": e, "rho": r["spearman_rho"],
                     "partial_rho": r.get("partial_rho_given_datacount")})
    return pd.DataFrame(rows)


def family_test(enrich, family=F_IN_CORE):
    """Mann-Whitney: are family elements more enriched than all other elements?"""
    inf = enrich[enrich.element.isin(family)]["log2_enrichment"]
    oth = enrich[~enrich.element.isin(family)]["log2_enrichment"]
    if len(inf) < 2 or len(oth) < 2:
        return {"n_family": len(inf), "note": "too few elements"}
    u = stats.mannwhitneyu(inf, oth, alternative="greater")
    return {"n_family": len(inf), "n_other": len(oth),
            "median_family": float(inf.median()), "median_other": float(oth.median()),
            "U": float(u.statistic), "p_one_sided": float(u.pvalue)}


# --------------------------------------------------------------------------------------
# 5. Test C — do proposals survive r2SCAN?
# --------------------------------------------------------------------------------------
def build_survival_table(df_meta, thermo_before, thermo_after, thr=0.0):
    """Joins 'before' (GGA-based hull) and 'after' (r2SCAN hull) energies above hull.
    stable_before: e_hull_before <= thr. lost: stable_before and e_hull_after > thr."""
    b = thermo_before[["material_id", "e_hull"]].rename(columns={"e_hull": "e_before"})
    a = thermo_after[["material_id", "e_hull"]].rename(columns={"e_hull": "e_after"})
    j = df_meta.merge(b, on="material_id").merge(a, on="material_id")
    j = j.dropna(subset=["e_before", "e_after"])
    j["stable_before"] = j["e_before"] <= thr + 1e-9
    j["lost"] = j["stable_before"] & (j["e_after"] > thr + 1e-9)
    return j


def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan, np.nan)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (p, c - h, c + h)


def loss_rates(surv, flag_col="has_f_in_core", groups=("GNoME", "MP_theoretical", "MP_known")):
    """Share of 'stable before' compounds that lose stability, by group x flag."""
    s = surv[surv["stable_before"]]
    rows = []
    for g in groups:
        for f in (True, False):
            d = s[(s["group"] == g) & (s[flag_col] == f)]
            p, lo, hi = wilson(int(d["lost"].sum()), len(d))
            rows.append({"group": g, flag_col: f, "n": len(d), "lost": int(d["lost"].sum()),
                         "rate": p, "ci_low": lo, "ci_high": hi})
    return pd.DataFrame(rows)


def mantel_haenszel_or(surv, flag_col="has_f_in_core", group="GNoME", strata_col="nelements"):
    """Odds ratio of losing stability for flagged vs not, within `group`, pooled over
    nelements strata (Mantel-Haenszel) with Robins-Breslow-Greenland 95% CI."""
    s = surv[(surv["stable_before"]) & (surv["group"] == group)]
    R = S = P_R = PS_QR = Q_S = 0.0
    for _, d in s.groupby(strata_col):
        a = ((d[flag_col]) & d["lost"]).sum(); b = ((d[flag_col]) & ~d["lost"]).sum()
        c = ((~d[flag_col]) & d["lost"]).sum(); dd = ((~d[flag_col]) & ~d["lost"]).sum()
        n = a + b + c + dd
        if n == 0 or (a + b) == 0 or (c + dd) == 0:
            continue
        r, s_ = a * dd / n, b * c / n
        p, q = (a + dd) / n, (b + c) / n
        R += r; S += s_
        P_R += p * r; PS_QR += p * s_ + q * r; Q_S += q * s_
    if R == 0 or S == 0:
        return {"or_mh": np.nan, "note": "not enough events"}
    orr = R / S
    se = math.sqrt(P_R / (2 * R * R) + PS_QR / (2 * R * S) + Q_S / (2 * S * S))
    return {"group": group, "flag": flag_col, "or_mh": orr,
            "ci_low": math.exp(math.log(orr) - 1.96 * se),
            "ci_high": math.exp(math.log(orr) + 1.96 * se)}


# --------------------------------------------------------------------------------------
# 6. Optional: artifact-risk classifier with honest evaluation
# --------------------------------------------------------------------------------------
def composition_features(df, elements):
    X = np.zeros((len(df), len(elements)))
    idx = {e: i for i, e in enumerate(elements)}
    for r, s in enumerate(df["composition"].values):
        comp = parse_composition(s)
        tot = sum(comp.values()) or 1
        for e, v in comp.items():
            if e in idx:
                X[r, idx[e]] = v / tot
    extra = df[["nelements", "nsites"]].astype(float).values
    return np.hstack([X, extra])


def evaluate_classifier(surv, group="GNoME", seed=0, n_splits=5, holdout_family=REE):
    """Predict 'lost' among stable_before compounds. Reports three evaluations:
      random K-fold (optimistic), GroupKFold by chemical system (honest),
      and train-without / test-on a held-out element family (hardest).
    Baselines: always-majority and the one-line rule 'contains f-in-core lanthanide'."""
    from sklearn.model_selection import KFold, GroupKFold
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.metrics import roc_auc_score, average_precision_score
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    s = surv[(surv["stable_before"]) & (surv["group"] == group)].reset_index(drop=True)
    y = s["lost"].astype(int).values
    if y.sum() < 10 or (len(y) - y.sum()) < 10:
        return pd.DataFrame([{"note": "too few positive/negative examples"}])
    els = sorted({e for l in s["elements"].apply(parse_elements) for e in l})
    X = composition_features(s, els)
    models = {
        "logistic": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0)),
        "grad_boost": lambda: HistGradientBoostingClassifier(random_state=seed, max_iter=300,
                                                             learning_rate=0.05),
    }
    rule = s["has_f_in_core"].astype(float).values
    rows = []

    def score(name, split, yt, p):
        if len(np.unique(yt)) < 2:
            return
        rows.append({"model": name, "split": split, "roc_auc": roc_auc_score(yt, p),
                     "pr_auc": average_precision_score(yt, p), "base_rate": yt.mean(),
                     "n_test": len(yt)})

    splits = {
        "random_kfold": list(KFold(n_splits, shuffle=True, random_state=seed).split(X)),
        "group_by_chemsys": list(GroupKFold(n_splits).split(X, y, s["chemsys"])),
    }
    fam = s["elements"].apply(lambda l: bool(set(parse_elements(l)) & set(holdout_family))).values
    splits["holdout_family"] = [(np.where(~fam)[0], np.where(fam)[0])]

    for split_name, folds in splits.items():
        for mname, mk in models.items():
            yt_all, p_all = [], []
            for tr, te in folds:
                if len(np.unique(y[tr])) < 2:
                    continue
                m = mk().fit(X[tr], y[tr])
                yt_all.append(y[te]); p_all.append(m.predict_proba(X[te])[:, 1])
            if yt_all:
                score(mname, split_name, np.concatenate(yt_all), np.concatenate(p_all))
        te_all = np.concatenate([te for _, te in folds])
        score("rule_f_in_core", split_name, y[te_all], rule[te_all])
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------------------
# 7. Stand-in data for testing the pipeline without an API key
# --------------------------------------------------------------------------------------
def make_synthetic(n_known=6000, n_theo=4000, n_gnome=8000, effect=True, seed=0):
    """Fake data with the same columns as the real pipeline. If effect=True, GNoME
    over-samples f-in-core lanthanides AND those compounds lose stability under
    r2SCAN more often. If effect=False, there is no planted signal: tests should
    come back null. ONLY for checking the code; never report these numbers."""
    rng = np.random.default_rng(seed)
    hhi = element_hhi_table()
    pool = [e for e in hhi if e not in RADIOACTIVE and e not in ("He",)]
    base_w = np.ones(len(pool))
    common = {"O": 8, "Li": 3, "Na": 3, "Mg": 3, "Al": 3, "Si": 3, "P": 3, "S": 3,
              "Fe": 3, "Cu": 2, "Zn": 2, "Ca": 3, "K": 2, "Ti": 2, "Mn": 2, "Co": 2, "Ni": 2}
    for i, e in enumerate(pool):
        base_w[i] *= common.get(e, 1.0)
    gn_w = base_w.copy()
    if effect:
        for i, e in enumerate(pool):
            if e in F_IN_CORE:
                gn_w[i] *= 3.0
    sens_el = {e: max(0.005, (0.12 if (effect and e in F_IN_CORE) else 0.04) + rng.normal(0, 0.01))
               for e in pool}

    def gen(n, group, w, nel_p):
        rows = []
        p = w / w.sum()
        for k in range(n):
            ne = rng.choice([1, 2, 3, 4, 5], p=nel_p)
            els = list(rng.choice(pool, size=ne, replace=False, p=p))
            comp = {e: float(rng.integers(1, 5)) for e in els}
            rows.append({"material_id": f"{group}-{k}", "formula": "".join(els),
                         "composition": ";".join(f"{e}:{v}" for e, v in comp.items()),
                         "elements": ",".join(sorted(els)), "nelements": ne,
                         "nsites": int(sum(comp.values()) * rng.integers(1, 4)),
                         "theoretical": group != "known", "group_raw": group})
        return rows
    rows = (gen(n_known, "known", base_w, [0.05, 0.35, 0.4, 0.15, 0.05]) +
            gen(n_theo, "theo", base_w, [0.02, 0.3, 0.45, 0.18, 0.05]) +
            gen(n_gnome, "gnome", gn_w, [0.01, 0.1, 0.35, 0.39, 0.15]))
    df = pd.DataFrame(rows)
    df["group"] = df["group_raw"].map({"known": "MP_known", "theo": "MP_theoretical",
                                       "gnome": "GNoME"})
    # energies
    ef_gga = rng.normal(-1.5, 0.6, len(df))
    s = df["elements"].apply(lambda l: np.mean([sens_el[e] for e in parse_elements(l)])).values
    shift = rng.normal(0, s)
    fcore = df["elements"].apply(lambda l: bool(set(parse_elements(l)) & set(F_IN_CORE))).values
    is_gn = (df["group"] == "GNoME").values
    e_before = np.where(is_gn, np.abs(rng.normal(0, 0.005, len(df))) * (rng.random(len(df)) < 0.3),
                        np.abs(rng.normal(0, 0.03, len(df))) * (rng.random(len(df)) < 0.5))
    extra = 0.02 + (0.08 if effect else 0.0) * (fcore & is_gn)
    e_after = np.clip(e_before + rng.normal(extra, 0.03, len(df)), 0, None) * (rng.random(len(df)) < 0.85)
    tg = pd.DataFrame({"material_id": df.material_id, "thermo_type": THERMO_GGA,
                       "e_hull": e_before, "ef": ef_gga})
    tr = pd.DataFrame({"material_id": df.material_id, "thermo_type": THERMO_R2,
                       "e_hull": e_after, "ef": ef_gga + shift})
    return df.drop(columns="group_raw"), tg, tr

In [ ]:
%%writefile crm_v2.py
"""
crm_v2.py — extended audit (Levels 1-4). Builds on crm_audit.py (v1).

Level 1  make the main result solid: two critical lists (EU 2023, US 2022), atom and
         mass fractions, duplicates removed, chemistry/crystal matching, a "uniform
         element choice" expectation, null calibration.
Level 2  explain the cause: gap decomposition (chemistry mix vs within-type choice),
         inheritance from computed data, near-copy templates, stability margins,
         r2SCAN re-check from DeepMind's own file.
Level 3  real-world relevance: Energy-GNoME application shortlists; non-GNoME
         high-throughput baselines from DeepMind's external hull file (OQMD, WBM, ...).
Level 4  own calculations: re-check a sample with a universal ML potential.
"""
from __future__ import annotations

import io
import os
import json
import math
import zipfile
from math import comb

import numpy as np
import pandas as pd
from scipy import stats

import crm_audit as ca

# --------------------------------------------------------------------------------------
# Lists and constants
# --------------------------------------------------------------------------------------
CRM_EU = list(ca.CRM_EU2023)                 # EU 2023 (element-mappable entries)
CRM_EU_AMBIG = list(ca.CRM_AMBIGUOUS)         # Al, F, C, Si, He
# US 2022 Final List of Critical Minerals (USGS, Federal Register 87 FR 10381).
CRM_US = ["Al", "Sb", "As", "Ba", "Be", "Bi", "Ce", "Cs", "Cr", "Co", "Dy", "Er", "Eu",
          "F", "Gd", "Ga", "Ge", "C", "Hf", "Ho", "In", "Ir", "La", "Li", "Lu", "Mg",
          "Mn", "Nd", "Ni", "Nb", "Pd", "Pt", "Pr", "Rh", "Rb", "Ru", "Sm", "Sc", "Ta",
          "Te", "Tb", "Tm", "Sn", "Ti", "W", "V", "Yb", "Y", "Zn", "Zr"]
US_AMBIG = ["Al", "F", "C"]                   # bauxite/aluminum, fluorspar, graphite
CRM_US_STRICT = [e for e in CRM_US if e not in US_AMBIG]
ANIONS = {"O", "S", "Se", "Te", "N", "F", "Cl", "Br", "I", "H"}
NOBLE = {"He", "Ne", "Ar", "Kr", "Xe", "Rn"}

GNOME_BASE = "https://storage.googleapis.com/gdm_materials_discovery/"
GNOME_SUMMARY_URL = GNOME_BASE + "gnome_data/stable_materials_summary.csv"
GNOME_R2SCAN_URL = GNOME_BASE + "gnome_data/stable_materials_r2scan.csv"
GNOME_BYID_ZIP_URL = GNOME_BASE + "gnome_data/by_id.zip"
GNOME_EXTERNAL_URL = GNOME_BASE + "external_data/external_materials_summary.csv"

EG_BASE = "https://raw.githubusercontent.com/paolodeangelis/Energy-GNoME/main/data/final/"
EG_FILES = ([("cathode", ion, f"cathodes/insertion/{ion}")
             for ion in ["Al", "Ca", "Cs", "K", "Li", "Mg", "Na", "Rb", "Y"]] +
            [("perovskite", "pure", "perovskites/pure_models"),
             ("perovskite", "mixed", "perovskites/mixed_models")] +
            [("thermoelectric", t, f"thermoelectrics/{t}")
             for t in ["300K", "430K", "560K", "690K", "820K", "950K"]])


# --------------------------------------------------------------------------------------
# Small helpers
# --------------------------------------------------------------------------------------
def cached_parquet(path, builder):
    if path and os.path.exists(path):
        return pd.read_parquet(path)
    df = builder()
    if path:
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        df.to_parquet(path, index=False)
    return df


def find_col(df, *keys, required=True):
    """Find a column by loose name match (case, spaces, underscores ignored)."""
    norm = {c: c.lower().replace(" ", "").replace("_", "").replace("(", "").replace(")", "")
            for c in df.columns}
    for k in keys:
        k = k.lower().replace(" ", "").replace("_", "")
        for c, n in norm.items():
            if n == k:
                return c
    for k in keys:
        k = k.lower().replace(" ", "").replace("_", "")
        for c, n in norm.items():
            if k in n:
                return c
    if required:
        raise KeyError(f"none of {keys} in {list(df.columns)}")
    return None


def comps_to_strings(comps):
    """pymatgen Compositions -> ('El:amt;...', 'A,B,C', nelements, reduced formula)."""
    cs, es, ns, rf = [], [], [], []
    for c in comps:
        d = {e.symbol: float(a) for e, a in c.items()}
        cs.append(";".join(f"{k}:{v:g}" for k, v in d.items()))
        es.append(",".join(sorted(d)))
        ns.append(len(d))
        rf.append(c.reduced_formula)
    return cs, es, ns, rf


def parse_formulas(formulas):
    from pymatgen.core import Composition
    out = []
    for f in formulas:
        try:
            out.append(Composition(str(f)))
        except Exception:
            out.append(None)
    return out


# --------------------------------------------------------------------------------------
# 1. Data loading
# --------------------------------------------------------------------------------------
def load_mp_compact(api_key, cache_dir, local_lake="/root/mp_lake"):
    """All Materials Project materials in one fast download (data lakehouse), reduced
    to the columns we need and saved as one small parquet file in cache_dir."""
    out = os.path.join(cache_dir, "mp_compact.parquet")

    def build():
        import glob
        import pyarrow as pa
        import pyarrow.dataset as pads
        from mp_api.client import MPRester
        files = [f for f in glob.glob(f"{local_lake}/**/*.parquet", recursive=True)
                 if "summary" in f and "_delta_log" not in f]
        if not files:
            with MPRester(api_key, local_dataset_cache=local_lake) as mpr:
                _ = mpr.materials.summary.search(fields=["material_id"], include_gnome=True)
            files = [f for f in glob.glob(f"{local_lake}/**/*.parquet", recursive=True)
                     if "summary" in f and "_delta_log" not in f]
        dset = pads.dataset(files, format="parquet")
        names = dset.schema.names
        want = [c for c in ["material_id", "deprecated", "theoretical", "nsites",
                            "formula_pretty", "energy_above_hull",
                            "formation_energy_per_atom", "symmetry"] if c in names]
        tab = dset.to_table(columns=want)
        df = tab.drop(["symmetry"]).to_pandas() if "symmetry" in want else tab.to_pandas()
        if "symmetry" in want:
            sym = tab.column("symmetry").combine_chunks()
            if pa.types.is_struct(sym.type):
                cs = sym.field("crystal_system").to_pandas()
            else:
                cs = pd.Series([(x or {}).get("crystal_system") for x in sym.to_pylist()])
            df["crystal_system"] = cs.astype(str).str.lower().values
        df["material_id"] = df["material_id"].astype(str)
        return df.drop_duplicates("material_id").reset_index(drop=True)
    return cached_parquet(out, build)


def mp_to_master(mp):
    """Materials Project compact table -> common schema with groups."""
    mp = mp[mp["deprecated"].astype(str) != "True"].copy()
    comps = parse_formulas(mp["formula_pretty"])
    ok = [c is not None for c in comps]
    mp, comps = mp[ok].copy(), [c for c in comps if c is not None]
    cs, es, ns, rf = comps_to_strings(comps)
    out = pd.DataFrame({
        "material_id": mp["material_id"].values, "formula": rf, "composition": cs,
        "elements": es, "nelements": ns, "nsites": mp["nsites"].values,
        "theoretical": mp["theoretical"].astype(bool).values,
        "crystal_system": mp.get("crystal_system", pd.Series(["unknown"] * len(mp))).values,
        "e_hull_mp": mp["energy_above_hull"].values})
    out["group"] = np.where(out["theoretical"], "MP_theoretical", "MP_known")
    return out


def load_gnome_raw(cache_dir, url=GNOME_SUMMARY_URL):
    return cached_parquet(os.path.join(cache_dir, "gnome_summary_raw.parquet"),
                          lambda: pd.read_csv(url))


def gnome_to_master(graw):
    comps = parse_formulas(graw[find_col(graw, "reducedformula")])
    keep = np.array([c is not None for c in comps])
    g = graw[keep].reset_index(drop=True)
    comps = [c for c in comps if c is not None]
    cs, es, ns, rf = comps_to_strings(comps)

    def opt(*k):
        c = find_col(g, *k, required=False)
        return g[c].values if c else np.full(len(g), np.nan)
    out = pd.DataFrame({
        "material_id": "gnome-" + g[find_col(g, "materialid")].astype(str).values,
        "gnome_id": g[find_col(g, "materialid")].astype(str).values,
        "formula": rf, "composition": cs, "elements": es, "nelements": ns,
        "nsites": g[find_col(g, "nsites")].values, "theoretical": True,
        "crystal_system": pd.Series(opt("crystalsystem")).astype(str).str.lower().values,
        "is_train": opt("istrain"),
        "e_decomp": opt("decompositionenergyperatom"),
        "e_decomp_all": opt("decompositionenergyperatomall"),
        "e_decomp_rel": opt("decompositionenergyperatomrelative"),
        "e_decomp_mp": opt("decompositionenergyperatommp"),
        "e_decomp_mp_oqmd": opt("decompositionenergyperatommpoqmd")})
    out["group"] = "GNoME"
    return out


def load_external(cache_dir, url=GNOME_EXTERNAL_URL):
    return cached_parquet(os.path.join(cache_dir, "gnome_external_raw.parquet"),
                          lambda: pd.read_csv(url))


def external_to_master(xraw):
    """DeepMind's external hull file (MP / OQMD / WBM / ... entries). Source is read
    from the id prefix when possible."""
    fcol = find_col(xraw, "reducedformula", "composition", "formula")
    comps = parse_formulas(xraw[fcol])
    keep = np.array([c is not None for c in comps])
    x = xraw[keep].reset_index(drop=True)
    comps = [c for c in comps if c is not None]
    cs, es, ns, rf = comps_to_strings(comps)
    idc = find_col(x, "materialid", "entryid", "id", required=False)
    ids = x[idc].astype(str) if idc else pd.Series([f"ext{i}" for i in range(len(x))])
    src = ids.str.extract(r"^([A-Za-z]+)", expand=False).fillna("other").str.lower()
    nsc = find_col(x, "nsites", required=False)
    out = pd.DataFrame({
        "material_id": "ext-" + ids.values, "formula": rf, "composition": cs,
        "elements": es, "nelements": ns,
        "nsites": x[nsc].values if nsc else np.nan, "theoretical": True,
        "crystal_system": "unknown", "source": src.values})
    out["group"] = "EXT_" + out["source"].str.upper()
    return out


def load_gnome_r2scan(cache_dir, url=GNOME_R2SCAN_URL):
    return cached_parquet(os.path.join(cache_dir, "gnome_r2scan_raw.parquet"),
                          lambda: pd.read_csv(url))


def load_energy_gnome(cache_dir):
    def build():
        import urllib.request
        rows = []
        for app, sub, path in EG_FILES:
            with urllib.request.urlopen(EG_BASE + path + "/candidates.json") as r:
                obj = json.loads(r.read().decode())
            d = pd.DataFrame(obj)
            d["application"], d["subset"] = app, sub
            rows.append(d)
        d = pd.concat(rows, ignore_index=True)
        for c in d.columns:            # keep every column; make text columns parquet-safe
            if d[c].dtype == object and c != "Material Id":
                num = pd.to_numeric(d[c], errors="coerce")
                d[c] = num if num.notna().mean() > 0.9 else d[c].astype(str)
        return d.astype({"Material Id": str})
    return cached_parquet(os.path.join(cache_dir, "energy_gnome_full.parquet"), build)


# --------------------------------------------------------------------------------------
# 2. Metrics
# --------------------------------------------------------------------------------------
def element_masses():
    from pymatgen.core import Element
    out = {}
    for el in Element:
        try:
            out[el.symbol] = float(el.atomic_mass)
        except Exception:
            pass
    return out


def load_hhi(override_csv=None):
    """Element HHI (production, reserves). Default: pymatgen (Gaultois et al. 2013,
    ~2011 USGS data). Override with a CSV: element,hhi_p[,hhi_r] (0-10000 scale)."""
    hhi = ca.element_hhi_table()
    if override_csv and os.path.exists(override_csv):
        o = pd.read_csv(override_csv)
        for _, r in o.iterrows():
            p = float(r["hhi_p"])
            rr = float(r["hhi_r"]) if "hhi_r" in o.columns and pd.notna(r["hhi_r"]) else hhi.get(r["element"], (p, p))[1]
            hhi[str(r["element"])] = (p, rr)
    return hhi


def add_metrics(df, hhi=None):
    """All per-compound measures, in one pass."""
    hhi = hhi or load_hhi()
    mass = element_masses()
    sets = {"eu": set(CRM_EU), "eu_amb": set(CRM_EU) | set(CRM_EU_AMBIG),
            "us": set(CRM_US_STRICT), "us_amb": set(CRM_US)}
    cols = {f"crm_{k}_{w}": [] for k in sets for w in ("atom", "mass")}
    cols.update({"hhi_p": [], "hhi_r": [], "hhi_p_mass": [], "hhi_p_max": []})
    for s in df["composition"].values:
        comp = ca.parse_composition(s)
        tot_a = sum(comp.values()) or 1.0
        tot_m = sum(v * mass.get(e, 0.0) for e, v in comp.items()) or 1.0
        for k, st in sets.items():
            a = sum(v for e, v in comp.items() if e in st)
            m = sum(v * mass.get(e, 0.0) for e, v in comp.items() if e in st)
            cols[f"crm_{k}_atom"].append(a / tot_a)
            cols[f"crm_{k}_mass"].append(m / tot_m)
        known = [(e, v) for e, v in comp.items() if e in hhi]
        ka = sum(v for _, v in known)
        km = sum(v * mass.get(e, 0.0) for e, v in known)
        cols["hhi_p"].append(sum(v * hhi[e][0] for e, v in known) / ka if ka else np.nan)
        cols["hhi_r"].append(sum(v * hhi[e][1] for e, v in known) / ka if ka else np.nan)
        cols["hhi_p_mass"].append(sum(v * mass.get(e, 0) * hhi[e][0] for e, v in known) / km if km else np.nan)
        cols["hhi_p_max"].append(max((hhi[e][0] for e, _ in known), default=np.nan))
    df = df.copy()
    for k, v in cols.items():
        df[k] = v
    df["crm_frac"] = df["crm_eu_atom"]           # name used by v1 functions
    els = df["elements"].str.split(",").apply(set)
    df["has_ree"] = els.apply(lambda e: bool(e & set(ca.REE)))
    df["has_f_in_core"] = els.apply(lambda e: bool(e & set(ca.F_IN_CORE)))
    df["has_u_element"] = els.apply(lambda e: bool(e & set(ca.U_ELEMENTS)))
    df["has_radioactive"] = els.apply(lambda e: bool(e & set(ca.RADIOACTIVE)))
    df["has_pgm"] = els.apply(lambda e: bool(e & {"Ru", "Rh", "Pd", "Os", "Ir", "Pt"}))
    df["chem_class"] = np.where(els.apply(lambda e: bool(e & ANIONS)), "anion", "intermetallic")
    df["chemsys"] = df["elements"].str.replace(",", "-")
    df["s_n"] = df["nelements"].astype(str)
    df["s_n_class"] = df["s_n"] + "_" + df["chem_class"]
    df["s_n_class_xtal"] = df["s_n_class"] + "_" + df["crystal_system"].astype(str)
    return df


def dedupe_by_formula(df, prefer_col=None):
    """One row per (group, reduced formula). Keeps the lowest prefer_col if given."""
    d = df
    if prefer_col and prefer_col in d.columns:
        d = d.sort_values(prefer_col, na_position="last")
    return d.drop_duplicates(["group", "formula"]).reset_index(drop=True)


# --------------------------------------------------------------------------------------
# 3. Level 1 analyses
# --------------------------------------------------------------------------------------
HEADLINE_METRICS = ["crm_eu_atom", "crm_eu_mass", "crm_us_atom", "crm_us_mass",
                    "has_ree", "has_f_in_core", "has_pgm", "hhi_p", "hhi_p_mass"]


def compare_grid(df, metrics, refs, strata_list, target="GNoME", n_boot=200, seed=0,
                 label=""):
    rows = []
    for ref in refs:
        if (df["group"] == ref).sum() == 0:
            continue
        for st in strata_list:
            for m in metrics:
                d = df[df["group"].isin([target, ref])]
                d = d[~d[st].astype(str).str.contains("unknown|nan|None", regex=True)] \
                    if st == "s_n_class_xtal" else d
                try:
                    r = ca.compare_metric(d.assign(**{m: d[m].astype(float)}), m, target, ref,
                                          strata_col=st, n_boot=n_boot, seed=seed)
                except (ZeroDivisionError, ValueError, IndexError):
                    continue          # no overlapping strata (e.g. no crystal data)
                rows.append(dict(check=label, ref=ref, matched_on=st, metric=m,
                                 target=r["mean_target"], ref_matched=r["mean_ref_matched"],
                                 ref_raw=r["mean_ref_raw"], diff=r["diff"],
                                 ci_low=r["ci_low"], ci_high=r["ci_high"],
                                 n_target=r["n_target"], n_ref=r["n_ref"]))
    return pd.DataFrame(rows)


def uniform_expectation(df, groups, family, label):
    """If each compound picked its k elements uniformly at random from the element
    pool, how often would it contain at least one member of `family`?
    Compared with what each group actually shows."""
    pool = sorted({e for s in df["elements"] for e in s.split(",")} - NOBLE)
    E, F = len(pool), len(set(family) & set(pool))
    p_none = {k: comb(E - F, k) / comb(E, k) for k in range(1, 12)}
    rows = []
    fam = set(family)
    for g in groups:
        d = df[df["group"] == g]
        if not len(d):
            continue
        k = d["nelements"].clip(upper=11).astype(int)
        expected = (1 - k.map(p_none)).mean()
        observed = d["elements"].str.split(",").apply(lambda e: bool(set(e) & fam)).mean()
        rows.append(dict(family=label, group=g, observed=observed, uniform_expected=expected,
                         ratio=observed / expected, pool_size=E, family_size=F))
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------------------
# 4. Level 2 analyses
# --------------------------------------------------------------------------------------
def decompose_gap(df, metric, target, ref, strata_col, n_boot=200, seed=0):
    """Splits the gap in `metric` between target and ref into:
       mix    = from having a different share of each stratum (e.g. more metal-only)
       within = from differences inside the same stratum.
    gap = sum_s wT(s)mT(s) - sum_s wR(s)mR(s)
        = sum_s wT(s)[mT(s)-mR(s)]  (within)  +  sum_s [wT(s)-wR(s)] mR(s)  (mix).
    Only strata present in both groups are used."""
    rng = np.random.default_rng(seed)
    d = df[df["group"].isin([target, ref])].dropna(subset=[metric])
    codes, uniq = pd.factorize(d[strata_col])
    isT = (d["group"] == target).values
    v = d[metric].astype(float).values
    K = len(uniq)

    def parts(idxT, idxR):
        nT = np.bincount(codes[idxT], minlength=K).astype(float)
        nR = np.bincount(codes[idxR], minlength=K).astype(float)
        sT = np.bincount(codes[idxT], weights=v[idxT], minlength=K)
        sR = np.bincount(codes[idxR], weights=v[idxR], minlength=K)
        ok = (nT > 0) & (nR > 0)
        wT, wR = nT[ok] / nT[ok].sum(), nR[ok] / nR[ok].sum()
        mT, mR = sT[ok] / nT[ok], sR[ok] / nR[ok]
        within = float((wT * (mT - mR)).sum())
        mix = float(((wT - wR) * mR).sum())
        return within + mix, mix, within
    iT, iR = np.where(isT)[0], np.where(~isT)[0]
    tot, mix, within = parts(iT, iR)
    bs = np.array([parts(rng.choice(iT, len(iT)), rng.choice(iR, len(iR)))
                   for _ in range(n_boot)])
    lo, hi = np.percentile(bs, [2.5, 97.5], axis=0)
    share = bs[:, 1] / np.where(bs[:, 0] == 0, np.nan, bs[:, 0])
    return dict(metric=metric, strata=strata_col, gap=tot, mix=mix, within=within,
                mix_share=mix / tot if tot else np.nan,
                mix_share_ci=(np.nanpercentile(share, 2.5), np.nanpercentile(share, 97.5)),
                gap_ci=(lo[0], hi[0]), mix_ci=(lo[1], hi[1]), within_ci=(lo[2], hi[2]))


def amplification(df, n_boot=50, seed=0):
    """Per element: enrichment of computed MP proposals vs known (an existing
    computational bias) against enrichment of GNoME vs known. Slope > 1 = GNoME
    amplifies a bias that already existed in computed data."""
    e1 = ca.element_enrichment(df, "MP_theoretical", "MP_known", n_boot=n_boot, seed=seed)
    e2 = ca.element_enrichment(df, "GNoME", "MP_known", n_boot=n_boot, seed=seed)
    j = e1[["element", "log2_enrichment"]].merge(
        e2[["element", "log2_enrichment"]], on="element", suffixes=("_mp_theo", "_gnome"))
    rho = stats.spearmanr(j["log2_enrichment_mp_theo"], j["log2_enrichment_gnome"]).statistic
    ts = stats.theilslopes(j["log2_enrichment_gnome"], j["log2_enrichment_mp_theo"])
    return dict(n_elements=len(j), spearman_rho=rho, theil_slope=ts.slope,
                slope_ci=(ts.low_slope, ts.high_slope)), j


def ln_templates(df, groups=("GNoME", "MP_known", "MP_theoretical")):
    LN = set(ca.REE)

    def key(s):
        c = ca.parse_composition(s)
        return ";".join(sorted(f"{'Ln' if e in LN else e}:{v:g}" for e, v in c.items()))
    rows = []
    for g in groups:
        d = df[(df["group"] == g) & df["has_ree"]]
        if not len(d):
            continue
        sizes = d["composition"].map(key).value_counts()
        rows.append(dict(group=g, ree_compounds=len(d), templates=len(sizes),
                         copies_per_template=len(d) / len(sizes),
                         share_templates_5plus=(sizes >= 5).mean(),
                         max_copies=int(sizes.max())))
    return pd.DataFrame(rows)


def margin_table(g, flags=("has_f_in_core", "has_ree", "has_u_element"),
                 margin_col="e_decomp_rel", thresholds=(0.010, 0.025)):
    """GNoME stability margins from DeepMind's own columns. margin = how far below the
    hull of all OTHER entries (meV/atom). 'Fragile' = margin below threshold.
    Compared between compounds with and without each flag (MH odds ratio over
    number of elements and chemistry type)."""
    d = g.dropna(subset=[margin_col]).copy()
    d["margin_meV"] = -1000 * d[margin_col].astype(float)
    rows = []
    for f in flags:
        for thr in thresholds:
            d["stable_before"] = True
            d["lost"] = d["margin_meV"] < thr * 1000
            for val in (True, False):
                s = d[d[f] == val]
                p, lo, hi = ca.wilson(int(s["lost"].sum()), len(s))
                rows.append(dict(flag=f, threshold_meV=thr * 1000, flag_value=val, n=len(s),
                                 median_margin_meV=s["margin_meV"].median(),
                                 fragile_share=p, ci_low=lo, ci_high=hi))
            o = ca.mantel_haenszel_or(d.assign(group="GNoME"), f, "GNoME", "s_n_class")
            rows.append(dict(flag=f, threshold_meV=thr * 1000, flag_value="odds_ratio",
                             n=len(d), fragile_share=o.get("or_mh"),
                             ci_low=o.get("ci_low"), ci_high=o.get("ci_high")))
    return pd.DataFrame(rows), d


def r2scan_check(r2raw, gmaster, thresholds=(0.001, 0.025)):
    """Uses DeepMind's r2SCAN file. Finds the id and a decomposition/hull column by
    name; if none is found it returns the column list so the analysis can be adapted."""
    idc = find_col(r2raw, "materialid", "material id", "id", required=False)
    cand = [c for c in r2raw.columns if any(k in c.lower() for k in
            ("decomposition", "hull", "e_above", "ehull"))]
    info = dict(columns=list(r2raw.columns), id_col=idc, stability_cols=cand, n=len(r2raw))
    if idc is None or not cand:
        return info, None, None
    col = cand[0]
    j = gmaster.merge(r2raw[[idc, col]].rename(columns={idc: "gnome_id", col: "e_r2"})
                      .astype({"gnome_id": str}), on="gnome_id", how="inner")
    info.update(used_col=col, matched=len(j))
    rows = []
    for thr in thresholds:
        j["stable_before"] = True
        j["lost"] = j["e_r2"].astype(float) > thr
        for f in ("has_f_in_core", "has_ree", "has_u_element"):
            for val in (True, False):
                s = j[j[f] == val]
                p, lo, hi = ca.wilson(int(s["lost"].sum()), len(s))
                rows.append(dict(flag=f, threshold_eV=thr, flag_value=val, n=len(s),
                                 unstable_share=p, ci_low=lo, ci_high=hi))
            o = ca.mantel_haenszel_or(j.assign(group="GNoME"), f, "GNoME", "s_n_class")
            rows.append(dict(flag=f, threshold_eV=thr, flag_value="odds_ratio", n=len(j),
                             unstable_share=o.get("or_mh"), ci_low=o.get("ci_low"),
                             ci_high=o.get("ci_high")))
    return info, pd.DataFrame(rows), j


# --------------------------------------------------------------------------------------
# 5. Level 3 analyses
# --------------------------------------------------------------------------------------
def energy_gnome_table(eg, master, metrics=("crm_eu_atom", "crm_us_atom", "has_ree",
                       "has_f_in_core", "has_pgm", "hhi_p"), conf_min=0.9, n_boot=200, seed=0):
    """Criticality of GNoME candidates shortlisted for real applications, compared with
    all GNoME and with known materials (matched on number of elements)."""
    g = master[master["group"] == "GNoME"]
    conf = "AI-experts confidence (-)"
    rows, joined = [], []
    for app in sorted(eg["application"].unique()):
        e = eg[eg["application"] == app]
        sets = [("all", e)]
        if conf in e.columns:
            sets.append((f"confidence>={conf_min}", e[e[conf].astype(float) >= conf_min]))
        for lab, es in sets:
            ids = set(es["Material Id"].astype(str))
            sub = g[g["gnome_id"].isin(ids)].assign(group=f"EG_{app}")
            if not len(sub):
                rows.append(dict(application=app, subset=lab, n_listed=len(ids), n_matched=0))
                continue
            joined.append(sub.assign(subset=lab))
            for m in metrics:
                both = pd.concat([sub, master[master["group"] == "MP_known"]])
                r = ca.compare_metric(both.assign(**{m: both[m].astype(float)}), m,
                                      f"EG_{app}", "MP_known", "s_n", n_boot=n_boot, seed=seed)
                rows.append(dict(application=app, subset=lab, n_listed=len(ids),
                                 n_matched=len(sub), metric=m, shortlist=r["mean_target"],
                                 all_gnome=float(g[m].astype(float).mean()),
                                 known_matched=r["mean_ref_matched"], diff_vs_known=r["diff"],
                                 ci_low=r["ci_low"], ci_high=r["ci_high"]))
    return pd.DataFrame(rows), (pd.concat(joined) if joined else pd.DataFrame())


# --------------------------------------------------------------------------------------
# 6. Level 4: re-check a sample with a universal ML potential
# --------------------------------------------------------------------------------------
def pick_level4_sample(master, n_per_group=40, seed=0, max_elements=3):
    """Ternary-or-smaller GNoME compounds; half with f-in-core lanthanides, half with
    no rare earth, same chemistry-type mix. Systems where GGA and GGA+U would be mixed
    (U-element together with O or F) are excluded to keep energies comparable."""
    rng = np.random.default_rng(seed)
    g = master[(master["group"] == "GNoME") & (master["nelements"] <= max_elements)].copy()
    els = g["elements"].str.split(",").apply(set)
    mixed_u = els.apply(lambda e: bool(e & set(ca.U_ELEMENTS)) and bool(e & {"O", "F"}))
    g = g[~mixed_u & ~g["has_radioactive"]]
    a = g[g["has_f_in_core"]]
    b = g[~g["has_ree"]]
    picks = []
    for cls in ("anion", "intermetallic"):
        aa, bb = a[a["chem_class"] == cls], b[b["chem_class"] == cls]
        k = min(n_per_group // 2, len(aa), len(bb))
        if k:
            picks.append(aa.sample(k, random_state=int(rng.integers(1e9))))
            picks.append(bb.sample(k, random_state=int(rng.integers(1e9))))
    return pd.concat(picks).reset_index(drop=True) if picks else g.head(0)


def subsystems(chemsys):
    els = chemsys.split("-")
    out = []
    for r in range(1, len(els) + 1):
        for c in __import__("itertools").combinations(sorted(els), r):
            out.append("-".join(c))
    return out


def gnome_structure_reader(url=GNOME_BYID_ZIP_URL, local_zip=None):
    """Returns a function gnome_id -> pymatgen Structure, reading single CIFs from the
    remote zip with HTTP range requests (no full download). Falls back to a local zip."""
    import fsspec
    from pymatgen.core import Structure
    fobj = open(local_zip, "rb") if local_zip else fsspec.open(url, "rb", block_size=2**20).open()
    z = zipfile.ZipFile(fobj)
    index = {}
    for n in z.namelist():
        base = os.path.basename(n)
        stem = base.rsplit(".", 1)[0]
        if base.lower().endswith(".cif"):
            index[stem] = n
    if not index:
        raise RuntimeError(f"No CIF files in zip. First names: {z.namelist()[:10]}")

    def read(gid):
        name = index.get(str(gid))
        if name is None:
            return None
        return Structure.from_str(z.read(name).decode(), fmt="cif")
    read.n_files = len(index)
    return read


def mp_structure_fetcher(api_key, e_hull_max=0.05):
    """Returns a function chemsys -> list of (material_id, Structure) for all MP
    entries in the chemical system and its subsystems near the hull."""
    from mp_api.client import MPRester
    from pymatgen.core import Structure

    def fetch(chemsys):
        with MPRester(api_key, use_document_model=False, mute_progress_bars=True) as mpr:
            docs = mpr.materials.summary.search(
                chemsys=subsystems(chemsys), energy_above_hull=(0, e_hull_max),
                deprecated=False, fields=["material_id", "structure"])
        out = []
        for d in docs:
            s = d["structure"]
            out.append((str(d["material_id"]), s if hasattr(s, "lattice") else Structure.from_dict(s)))
        return out
    return fetch


def make_calculator(name="mace", device=None):
    if name == "emt":                       # only for testing the code
        from ase.calculators.emt import EMT
        return EMT()
    if device is None:
        try:
            import torch
            device = "cuda" if torch.cuda.is_available() else "cpu"
        except Exception:
            device = "cpu"
    if name == "mace":
        from mace.calculators import mace_mp
        return mace_mp(model="medium", device=device, default_dtype="float32")
    if name == "chgnet":
        from chgnet.model.dynamics import CHGNetCalculator
        return CHGNetCalculator(use_device=device)
    raise ValueError(name)


def relax_energy(structure, calc, fmax=0.05, steps=300):
    from pymatgen.io.ase import AseAtomsAdaptor
    try:
        from ase.filters import FrechetCellFilter as CellFilter
    except Exception:
        from ase.constraints import ExpCellFilter as CellFilter
    from ase.optimize import FIRE
    atoms = AseAtomsAdaptor.get_atoms(structure)
    atoms.calc = calc
    opt = FIRE(CellFilter(atoms), logfile=None)
    opt.run(fmax=fmax, steps=steps)
    return float(atoms.get_potential_energy()), len(atoms), int(opt.nsteps)


def run_level4(sample, gnome_reader, mp_fetcher, calc, cache_path, max_mp_per_system=60,
               log_every=10):
    """Relaxes each sampled GNoME compound and the MP phases of its chemical system
    with the same ML potential, then measures how far the GNoME compound lies below
    (negative) or above (positive) the ML-potential hull of the MP phases.
    Energies are checkpointed to cache_path so an interrupted run can resume."""
    from pymatgen.analysis.phase_diagram import PhaseDiagram, PDEntry
    done = pd.read_parquet(cache_path) if os.path.exists(cache_path) else \
        pd.DataFrame(columns=["key", "formula", "energy", "natoms", "steps", "ok"])
    E = {r.key: r for r in done.itertuples()}
    new_rows = []

    def energy_of(key, struct):
        if key in E:
            return E[key]
        try:
            e, n, st = relax_energy(struct, calc)
            row = dict(key=key, formula=struct.composition.formula, energy=e, natoms=n,
                       steps=st, ok=bool(np.isfinite(e)))
        except Exception as ex:
            row = dict(key=key, formula=struct.composition.formula, energy=np.nan,
                       natoms=len(struct), steps=-1, ok=False)
        new_rows.append(row)
        E[key] = pd.Series(row)
        return E[key]

    def checkpoint():
        if new_rows:
            cur = pd.DataFrame([dict(key=k, formula=v.formula, energy=v.energy, natoms=v.natoms,
                                     steps=v.steps, ok=v.ok) for k, v in E.items()])
            os.makedirs(os.path.dirname(cache_path) or ".", exist_ok=True)
            cur.to_parquet(cache_path, index=False)
            new_rows.clear()

    results = []
    for i, r in enumerate(sample.itertuples()):
        s = gnome_reader(r.gnome_id)
        if s is None:
            results.append(dict(gnome_id=r.gnome_id, status="no_structure"))
            continue
        g = energy_of(f"gnome:{r.gnome_id}", s)
        mp = mp_fetcher(r.chemsys)[:max_mp_per_system]
        entries = []
        for mid, ms in mp:
            m = energy_of(f"mp:{mid}", ms)
            if m.ok:
                entries.append(PDEntry(ms.composition, float(m.energy)))
        elems = set(r.elements.split(","))
        have = {e.symbol for en in entries for e in en.composition.elements
                if len(en.composition.elements) == 1}
        if not g.ok or have != elems:
            results.append(dict(gnome_id=r.gnome_id, status="incomplete_references"))
        else:
            pdg = PhaseDiagram(entries)
            cand = PDEntry(s.composition, float(g.energy))
            _, e_ml = pdg.get_decomp_and_e_above_hull(cand, allow_negative=True)
            results.append(dict(gnome_id=r.gnome_id, status="ok", e_hull_ml=float(e_ml),
                                n_mp_phases=len(entries)))
        if (i + 1) % log_every == 0:
            checkpoint()
            ok = sum(1 for x in results if x.get("status") == "ok")
            print(f"  {i + 1}/{len(sample)} compounds done ({ok} ok)")
    checkpoint()
    res = pd.DataFrame(results).merge(sample, on="gnome_id", how="left")
    return res


def level4_summary(res, thresholds=(0.0, 0.025, 0.05)):
    d = res[res["status"] == "ok"].copy()
    rows = []
    for thr in thresholds:
        d["stable_before"] = True
        d["lost"] = d["e_hull_ml"] > thr
        for val in (True, False):
            s = d[d["has_f_in_core"] == val]
            p, lo, hi = ca.wilson(int(s["lost"].sum()), len(s))
            rows.append(dict(threshold_eV=thr, has_f_in_core=val, n=len(s),
                             not_stable_share=p, ci_low=lo, ci_high=hi,
                             median_e_hull_ml=s["e_hull_ml"].median()))
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------------------
# 7. Demo data in the REAL file layouts (to test the whole notebook without a key)
# --------------------------------------------------------------------------------------
def make_demo_sources(seed=0, n_mp=6000, n_gnome=8000):
    """Fake inputs with the same columns as the real files, with a planted effect:
    GNoME favours f-in-core lanthanides and metal-only compounds, and its
    f-in-core compounds have thinner stability margins. ONLY for testing code."""
    rng = np.random.default_rng(seed)
    hhi = ca.element_hhi_table()
    pool = [e for e in hhi if e not in ca.RADIOACTIVE and e not in NOBLE]
    w = np.array([8 if e in ANIONS else 1.0 for e in pool])
    wg = np.array([2 if e in ANIONS else (3.0 if e in ca.F_IN_CORE else 1.0) for e in pool])
    xtal = ["cubic", "tetragonal", "orthorhombic", "hexagonal", "trigonal", "monoclinic", "triclinic"]

    def formulas(n, weights, nel_p):
        out = []
        p = weights / weights.sum()
        for _ in range(n):
            k = rng.choice([1, 2, 3, 4, 5], p=nel_p)
            els = rng.choice(pool, size=k, replace=False, p=p)
            out.append("".join(f"{e}{int(rng.integers(1, 5))}" for e in els))
        return out
    f_mp = formulas(n_mp, w, [0.05, 0.35, 0.4, 0.15, 0.05])
    theo = rng.random(n_mp) < 0.65
    mp = pd.DataFrame({"material_id": [f"mp-{i}" for i in range(n_mp)], "deprecated": False,
                       "theoretical": theo, "nsites": rng.integers(2, 60, n_mp),
                       "formula_pretty": f_mp, "energy_above_hull": np.abs(rng.normal(0, .05, n_mp)),
                       "formation_energy_per_atom": rng.normal(-1.5, .6, n_mp),
                       "crystal_system": rng.choice(xtal, n_mp)})
    f_g = formulas(n_gnome, wg, [0.0, 0.1, 0.4, 0.38, 0.12])
    fc = np.array([any(e in f for e in ca.F_IN_CORE) for f in f_g])  # crude flag
    margin = np.abs(rng.normal(0.06, 0.04, n_gnome)) * np.where(fc, 0.5, 1.0)
    graw = pd.DataFrame({
        "Composition": f_g, "MaterialId": [f"{i:010x}" for i in range(n_gnome)],
        "Reduced Formula": f_g, "Elements": "", "NSites": rng.integers(4, 80, n_gnome),
        "Crystal System": rng.choice(xtal, n_gnome), "Is Train": rng.random(n_gnome) < 0.3,
        "Decomposition Energy Per Atom": -margin, "Decomposition Energy Per Atom All": -margin,
        "Decomposition Energy Per Atom Relative": -margin,
        "Decomposition Energy Per Atom MP": -margin - 0.02,
        "Decomposition Energy Per Atom MP OQMD": -margin - 0.01})
    r2 = pd.DataFrame({"MaterialId": graw["MaterialId"].sample(3000, random_state=seed).values})
    fc_r2 = graw.set_index("MaterialId").loc[r2["MaterialId"], "Reduced Formula"].apply(
        lambda f: any(e in f for e in ca.F_IN_CORE)).values
    r2["Decomposition Energy Per Atom"] = rng.normal(0.0, 0.03, len(r2)) + 0.03 * fc_r2
    ext_f = formulas(4000, w * np.array([1.5 if e in ca.REE else 1 for e in pool]),
                     [0.05, 0.3, 0.45, 0.15, 0.05])
    xraw = pd.DataFrame({"MaterialId": [(f"mp-{i}" if i % 2 else f"{i:012x}") for i in range(4000)],
                         "Reduced Formula": ext_f, "NSites": rng.integers(2, 40, 4000)})
    ids = graw["MaterialId"].values
    eg = pd.DataFrame({"Material Id": rng.choice(ids, 1500, replace=False),
                       "AI-experts confidence (-)": rng.random(1500),
                       "application": rng.choice(["cathode", "perovskite", "thermoelectric"], 1500),
                       "subset": "demo"})
    thermo_rows = []
    for t, shift in ((ca.THERMO_GGA, 0.0), (ca.THERMO_R2, 0.05)):
        thermo_rows.append(pd.DataFrame({"material_id": mp["material_id"], "thermo_type": t,
                                         "e_hull": np.clip(mp["energy_above_hull"] + rng.normal(0, shift, n_mp), 0, None),
                                         "is_stable": False,
                                         "ef": mp["formation_energy_per_atom"] + rng.normal(0, shift, n_mp)}))
    thermo = {ca.THERMO_GGA: thermo_rows[0], ca.THERMO_R2: thermo_rows[1]}
    return mp, graw, r2, xraw, eg, thermo

In [ ]:
%%writefile crm_v3.py
"""
crm_v3.py — two extensions on top of crm_v2.py.

A. Country exposure. Uses USGS Mineral Commodity Summaries 2025 world production data
   (production by country, latest year) to measure, per compound, how much of its
   supply depends on single countries. Gives an up-to-date HHI as well.

B. Lower-risk alternatives. For GNoME candidates shortlisted for real applications
   (Energy-GNoME), asks: for a high-risk candidate, is there another candidate with
   similar predicted performance but lower supply risk? And how much performance is
   lost if we only keep low-risk candidates?
"""
from __future__ import annotations

import io
import os
import re
import json
import zipfile
import urllib.request

import numpy as np
import pandas as pd

import crm_audit as ca
import crm_v2 as v2

# ======================================================================================
# A. Country exposure
# ======================================================================================
USGS_WORLD_ITEM = "6798fd34d34ea8c18376e8ee"   # MCS 2025 World Production, Capacity, Reserves
CHINA = "China"

# element -> ordered list of (commodity keyword, preferred TYPE keyword or None).
# The first commodity found in the file is used. "proxy" marks elements that borrow
# another commodity's country pattern (stated in the paper's methods).
ELEMENT_SOURCES = {
    "Al": [("BAUXITE", "mine"), ("ALUMINUM", None)], "Sb": [("ANTIMONY", "mine")],
    "As": [("ARSENIC", None)], "Ba": [("BARITE", "mine")], "Be": [("BERYLLIUM", "mine")],
    "Bi": [("BISMUTH", None)], "B": [("BORON", None)], "Br": [("BROMINE", None)],
    "Cd": [("CADMIUM", None)], "Cr": [("CHROMIUM", None)], "Co": [("COBALT", "mine")],
    "Cu": [("COPPER", "mine")], "F": [("FLUORSPAR", None)], "Ga": [("GALLIUM", None)],
    "Ge": [("GERMANIUM", None)], "Au": [("GOLD", "mine")], "C": [("GRAPHITE", None)],
    "In": [("INDIUM", None)], "I": [("IODINE", None)], "Fe": [("IRON ORE", None)],
    "Pb": [("LEAD", "mine")], "Li": [("LITHIUM", "mine")],
    "Mg": [("MAGNESIUM METAL", None), ("MAGNESIUM", None)], "Mn": [("MANGANESE", "mine")],
    "Hg": [("MERCURY", "mine")], "Mo": [("MOLYBDENUM", "mine")], "Ni": [("NICKEL", "mine")],
    "Nb": [("NIOBIUM", "mine")], "N": [("NITROGEN", None)], "P": [("PHOSPHATE", "mine")],
    "Pt": [("PLATINUM", "platinum")], "Pd": [("PLATINUM", "palladium")],
    "Ru": [("PLATINUM", "platinum")], "Rh": [("PLATINUM", "platinum")],      # proxy
    "Ir": [("PLATINUM", "platinum")], "Os": [("PLATINUM", "platinum")],      # proxy
    "K": [("POTASH", None)], "Re": [("RHENIUM", None)], "Se": [("SELENIUM", None)],
    "Si": [("SILICON", None)], "Ag": [("SILVER", "mine")], "Na": [("SALT", None)],
    "Cl": [("SALT", None)], "Sr": [("STRONTIUM", None)], "S": [("SULFUR", None)],
    "Ta": [("TANTALUM", "mine")], "Te": [("TELLURIUM", None)], "Sn": [("TIN", "mine")],
    "Ti": [("TITANIUM MINERAL", "ilmenite"), ("TITANIUM MINERAL", None), ("TITANIUM", None)],
    "W": [("TUNGSTEN", "mine")], "V": [("VANADIUM", "mine")], "Zn": [("ZINC", "mine")],
    "Zr": [("ZIRCONIUM", None)], "Hf": [("ZIRCONIUM", None)],                 # proxy
    "Y": [("YTTRIUM", None), ("RARE EARTH", "mine")],
}
for _e in ca.LANTHANIDES + ["Sc"]:
    ELEMENT_SOURCES.setdefault(_e, [("RARE EARTH", "mine")])               # proxy per element
PROXY_ELEMENTS = {"Ru", "Rh", "Ir", "Os", "Hf", "Sc"} | set(ca.LANTHANIDES)


def fetch_usgs_world(cache_dir, item_id=USGS_WORLD_ITEM, local_file=None):
    """Download the MCS 2025 world data table (CSV, possibly inside a zip) from
    ScienceBase, or read a file you uploaded yourself (local_file)."""
    out = os.path.join(cache_dir, "usgs_mcs2025_world_raw.parquet")
    if os.path.exists(out):
        return pd.read_parquet(out)
    if local_file:
        raw = _read_any_table(open(local_file, "rb").read(), local_file)
    else:
        meta = json.loads(urllib.request.urlopen(
            f"https://www.sciencebase.gov/catalog/item/{item_id}?format=json", timeout=60).read())
        files = list(meta.get("files", []))
        for fc in meta.get("facets", []) or []:
            files += fc.get("files", [])
        print("ScienceBase files:", [f.get("name") for f in files])
        pick = [f for f in files if re.search(r"world", f.get("name", ""), re.I)
                and f.get("name", "").lower().endswith((".csv", ".zip", ".xlsx"))]
        pick = pick or [f for f in files if f.get("name", "").lower().endswith((".csv", ".zip"))]
        if not pick:
            raise RuntimeError("No CSV/ZIP found; download the world data file by hand and set USGS_LOCAL_FILE.")
        f = pick[0]
        url = f.get("downloadUri") or f.get("url")
        print("downloading", f.get("name"), "from", url)
        raw = _read_any_table(urllib.request.urlopen(url, timeout=300).read(), f.get("name", ""))
    raw = raw.astype(str)
    raw.to_parquet(out, index=False)
    return raw


def _read_any_table(blob, name):
    name = name.lower()
    if name.endswith(".zip"):
        z = zipfile.ZipFile(io.BytesIO(blob))
        cands = [n for n in z.namelist() if n.lower().endswith(".csv")]
        world = [n for n in cands if "world" in n.lower()] or cands
        print("zip members used:", world[:3])
        return pd.concat([_read_csv_bytes(z.read(n)) for n in world], ignore_index=True)
    if name.endswith(".xlsx"):
        return pd.read_excel(io.BytesIO(blob))
    return _read_csv_bytes(blob)


def _read_csv_bytes(b):
    for enc in ("utf-8", "latin-1"):
        try:
            return pd.read_csv(io.BytesIO(b), encoding=enc, dtype=str)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(io.BytesIO(b), encoding="latin-1", dtype=str, on_bad_lines="skip")


def _num(x):
    s = re.sub(r"[,\s]", "", str(x))
    s = re.sub(r"[^\d.\-eE]", "", s) if re.search(r"\d", s) else ""
    try:
        return float(s)
    except ValueError:
        return np.nan


def parse_usgs_world(raw):
    """-> long table: commodity, type, country, value (latest production year)."""
    cols = {c: re.sub(r"[^a-z0-9]", "", c.lower()) for c in raw.columns}
    def find(*keys):
        for k in keys:
            for c, n in cols.items():
                if k in n:
                    return c
        return None
    c_comm, c_country, c_type = find("commodity"), find("country"), find("type")
    prod = [(int(m.group()), c) for c, n in cols.items()
            if ("prod" in n) and (m := re.search(r"20\d\d", n))]
    if not (c_comm and c_country and prod):
        raise KeyError(f"Unexpected USGS columns: {list(raw.columns)}")
    year, c_val = max(prod)
    long = pd.DataFrame({"commodity": raw[c_comm].astype(str).str.upper().str.strip(),
                         "type": raw[c_type].astype(str).str.strip() if c_type else "",
                         "country": raw[c_country].astype(str).str.strip(),
                         "value": raw[c_val].map(_num)})
    long.attrs["year_column"] = c_val
    return long


def element_country_shares(long):
    """Per element: country shares of world production, plus summary
    (new HHI 0-10000, top country, top share, China share)."""
    shares, summary = [], []
    is_world = lambda s: s.str.contains("world", case=False)
    is_other = lambda s: s.str.contains("other", case=False)
    for el, sources in ELEMENT_SOURCES.items():
        used = None
        for kw, tkw in sources:
            d = long[long["commodity"].str.contains(kw, regex=False)]
            if tkw and len(d) and d["type"].str.contains(tkw, case=False).any():
                d = d[d["type"].str.contains(tkw, case=False)]
            elif len(d) and d["type"].nunique() > 1:
                mine = d["type"].str.contains("mine", case=False)
                d = d[mine] if mine.any() else d[d["type"] == d["type"].iloc[0]]
            d = d.dropna(subset=["value"])
            if len(d[~is_world(d["country"])]):
                used = (kw, tkw, d)
                break
        if used is None:
            summary.append(dict(element=el, source=None))
            continue
        kw, tkw, d = used
        world = d.loc[is_world(d["country"]), "value"]
        cty = d[~is_world(d["country"])].groupby("country", as_index=False)["value"].sum()
        total = max(world.max() if len(world) else 0, cty["value"].sum())
        if total <= 0:
            summary.append(dict(element=el, source=kw))
            continue
        cty["share"] = cty["value"] / total
        named = cty[~is_other(cty["country"])]
        hhi = float((100 * named["share"]) .pow(2).sum())
        top = named.sort_values("share", ascending=False).iloc[0]
        china = named.loc[named["country"].str.contains(CHINA, case=False), "share"].sum()
        summary.append(dict(element=el, source=kw, type_hint=tkw, proxy=el in PROXY_ELEMENTS,
                            hhi_new=hhi, top_country=top["country"], top_share=float(top["share"]),
                            china_share=float(china), n_countries=len(named)))
        shares.append(cty.assign(element=el)[["element", "country", "share"]])
    return pd.DataFrame(summary), (pd.concat(shares, ignore_index=True) if shares else pd.DataFrame())


def add_country_metrics(df, summary, conc_threshold=0.5):
    """Per compound (atom-weighted over elements with data):
    hhi_new, china_frac, top_share_max (weakest link), conc_frac = atom share from
    elements where one country produces >= conc_threshold, coverage."""
    s = summary.dropna(subset=["hhi_new"]).set_index("element")
    H, CH, TOP = s["hhi_new"].to_dict(), s["china_share"].to_dict(), s["top_share"].to_dict()
    out = {k: [] for k in ("hhi_new", "china_frac", "top_share_max", "conc_frac", "country_coverage")}
    for comp in df["composition"].values:
        c = ca.parse_composition(comp)
        tot = sum(c.values()) or 1.0
        kn = {e: v for e, v in c.items() if e in H}
        ka = sum(kn.values())
        out["country_coverage"].append(ka / tot)
        if ka == 0:
            for k in ("hhi_new", "china_frac", "top_share_max", "conc_frac"):
                out[k].append(np.nan)
            continue
        out["hhi_new"].append(sum(v * H[e] for e, v in kn.items()) / ka)
        out["china_frac"].append(sum(v * CH[e] for e, v in kn.items()) / ka)
        out["top_share_max"].append(max(TOP[e] for e in kn))
        out["conc_frac"].append(sum(v for e, v in kn.items() if TOP[e] >= conc_threshold) / ka)
    df = df.copy()
    for k, v in out.items():
        df[k] = v
    return df


def country_profile(df, shares, groups, ref="MP_known", target="GNoME", strata_col="s_n_class", top=12):
    """Average dependence on each country (share of a group's supply, atom-weighted over
    elements with data). The reference group is re-weighted to the target's strata mix."""
    S = shares.pivot_table(index="element", columns="country", values="share", aggfunc="sum").fillna(0)
    els = list(S.index)
    idx = {e: i for i, e in enumerate(els)}
    t_mix = df.loc[df["group"] == target, strata_col].value_counts(normalize=True)
    res = {}
    for g in groups:
        d = df[df["group"] == g]
        if not len(d):
            continue
        if g != target:
            r = d[strata_col].value_counts(normalize=True)
            w = d[strata_col].map(lambda s: t_mix.get(s, 0) / r[s]).values
        else:
            w = np.ones(len(d))
        acc = np.zeros(len(els)); wsum = 0.0
        for comp, wi in zip(d["composition"].values, w):
            if wi == 0:
                continue
            c = ca.parse_composition(comp)
            kn = {e: v for e, v in c.items() if e in idx}
            ka = sum(kn.values())
            if ka == 0:
                continue
            for e, v in kn.items():
                acc[idx[e]] += wi * v / ka
            wsum += wi
        res[g] = (acc / wsum) @ S.values if wsum else np.zeros(S.shape[1])
    prof = pd.DataFrame(res, index=S.columns)
    prof = prof.loc[prof.max(axis=1).sort_values(ascending=False).index[:top]]
    return prof


# ======================================================================================
# B. Lower-risk alternatives among application shortlists
# ======================================================================================
# application -> (performance column, mode, parameter). mode "higher": bigger is better,
# compared with a relative tolerance; "target": closeness to a target value (eV),
# compared with an absolute tolerance.
APP_PERF = {
    "cathode": ("Gravimetric energy (Wh/kg)", "higher", None),
    "thermoelectric": ("Average zT (-)", "higher", None),
    "perovskite": ("Average Band Gap (eV)", "target", 1.34),   # single-junction solar optimum
}
CONF_COL = "AI-experts confidence (-)"


def eg_join(eg, master, risk_cols=("crm_eu_atom", "crm_us_atom", "has_ree", "hhi_p",
                                   "hhi_new", "china_frac", "top_share_max", "conc_frac")):
    g = master[master["group"] == "GNoME"]
    keep = ["gnome_id", "formula", "elements", "nelements"] + [c for c in risk_cols if c in g.columns]
    j = eg.merge(g[keep], left_on="Material Id", right_on="gnome_id", how="inner")
    return j


def _performance(d, app):
    col, mode, par = APP_PERF[app]
    if col not in d.columns:
        return None
    x = pd.to_numeric(d[col], errors="coerce")
    return x if mode == "higher" else -(x - par).abs()


def alternatives(egj, risk="crm_eu_atom", high_risk=0.5, rel_tol=0.10, abs_tol=0.10,
                 min_risk_drop=0.20, conf_min=0.5):
    """For each application subset:
      - share of high-risk candidates (risk >= high_risk) that have at least one
        alternative with performance within tolerance AND risk lower by >= min_risk_drop;
      - share of Pareto-optimal candidates (best performance-risk trade-off) that are high-risk;
      - cost of choosing low-risk only: best and top-10 mean performance, all vs low-risk.
    Returns (summary table, per-candidate table with Pareto flags)."""
    rows, per = [], []
    for (app, sub), d in egj.groupby(["application", "subset"]):
        if app not in APP_PERF:
            continue
        d = d.copy()
        d["perf"] = _performance(d, app)
        if d["perf"] is None or d["perf"].isna().all():
            continue
        if CONF_COL in d.columns:
            d = d[pd.to_numeric(d[CONF_COL], errors="coerce") >= conf_min]
        d = d.dropna(subset=["perf", risk]).drop_duplicates("gnome_id")
        if len(d) < 5:
            continue
        mode = APP_PERF[app][1]
        p, r = d["perf"].values.astype(float), d[risk].values.astype(float)
        order = np.argsort(-p)
        ps, rs = p[order], r[order]
        prefmin = np.minimum.accumulate(rs)                       # lowest risk among >= perf
        hr = r >= high_risk
        thr = p - (rel_tol * np.abs(p) if mode == "higher" else abs_tol)
        k = np.searchsorted(-ps, -thr, side="right") - 1         # last index with ps >= thr
        has_alt = (k >= 0) & (prefmin[np.clip(k, 0, None)] <= r - min_risk_drop)
        # Pareto front: no other candidate with >= perf and < risk
        front_sorted = rs <= np.minimum.accumulate(np.r_[np.inf, rs[:-1]]) - 1e-12
        front = np.zeros(len(d), bool); front[order] = front_sorted
        low = ~hr
        best_all, best_low = ps[0], (p[low].max() if low.any() else np.nan)
        top_all = np.sort(p)[::-1][:10].mean()
        top_low = np.sort(p[low])[::-1][:10].mean() if low.sum() >= 1 else np.nan
        rows.append(dict(application=app, subset=sub, n=len(d), high_risk_share=hr.mean(),
                         high_risk_with_alternative=has_alt[hr].mean() if hr.any() else np.nan,
                         pareto_n=int(front.sum()), pareto_high_risk_share=hr[front].mean(),
                         best_perf_all=best_all, best_perf_low_risk=best_low,
                         top10_mean_all=top_all, top10_mean_low_risk=top_low,
                         top10_loss=((top_all - top_low) / abs(top_all) if mode == "higher"
                                     else top_all - top_low),   # % for higher-better; eV for target
                         loss_unit="fraction" if mode == "higher" else "eV from 1.34",
                         perf_metric=APP_PERF[app][0] + (" (closeness to 1.34 eV)" if mode == "target" else "")))
        per.append(d.assign(pareto=front, high_risk=hr, has_alternative=has_alt))
    return pd.DataFrame(rows), (pd.concat(per, ignore_index=True) if per else pd.DataFrame())


# ======================================================================================
# Demo inputs in the assumed real layouts (testing only)
# ======================================================================================
def make_demo_usgs(seed=0):
    rng = np.random.default_rng(seed)
    countries = ["China", "Australia", "Chile", "Congo (Kinshasa)", "Russia", "South Africa",
                 "Brazil", "United States", "Indonesia", "Canada", "Other countries"]
    comms = sorted({kw for v in ELEMENT_SOURCES.values() for kw, _ in v})
    rows = []
    for cm in comms:
        tp = "Mine production" if cm not in ("GALLIUM", "GERMANIUM", "SILICON") else "Refinery production"
        types = [("Mine production, platinum", 1), ("Mine production, palladium", 1)] if cm == "PLATINUM" else [(tp, 1)]
        for t, _ in types:
            china_bias = 8 if cm in ("RARE EARTH", "GALLIUM", "GERMANIUM", "GRAPHITE", "TUNGSTEN") else 1
            vals = rng.gamma(1.0, 100, len(countries)); vals[0] *= china_bias
            for c, v in zip(countries, vals):
                rows.append({"Source": "MCS2025", "COMMODITY": cm.title() if cm != "RARE EARTH" else "Rare earths",
                             "COUNTRY": c, "TYPE": t, "UNIT_MEAS": "metric tons",
                             "PROD_2023": f"{v * 0.95:,.0f}", "PROD_EST_ 2024": f"{v:,.0f}"})
            rows.append({"Source": "MCS2025", "COMMODITY": cm.title(), "COUNTRY": "World total (rounded)", "TYPE": t,
                         "UNIT_MEAS": "metric tons", "PROD_2023": "", "PROD_EST_ 2024": f"{vals.sum():,.0f}"})
    return pd.DataFrame(rows)


def add_demo_properties(eg, master, seed=0):
    """Gives the demo shortlist realistic property columns (weakly linked to risk)."""
    rng = np.random.default_rng(seed)
    g = master[master.group == "GNoME"].set_index("gnome_id")["crm_eu_atom"]
    risk = eg["Material Id"].map(g).fillna(0.5).values
    eg = eg.copy()
    eg["Gravimetric energy (Wh/kg)"] = np.where(eg.application == "cathode", rng.gamma(4, 120, len(eg)) * (1 + 0.3 * risk), np.nan)
    eg["Average zT (-)"] = np.where(eg.application == "thermoelectric", rng.gamma(3, 0.12, len(eg)) * (1 + 0.3 * risk), np.nan)
    eg["Average Band Gap (eV)"] = np.where(eg.application == "perovskite", rng.uniform(0.5, 3, len(eg)), np.nan)
    eg["subset"] = np.where(eg.application == "cathode", rng.choice(["Li", "Na"], len(eg)), "demo")
    return eg

## 2. Settings

In [ ]:
DEMO_MODE = False        # True = fake data to test the pipeline. False = real data.
SAVE_TO_DRIVE = True    # cache downloads on Google Drive
RUN_LEVEL4 = False      # True = run the ML-potential re-check (needs a GPU runtime, 1-3 h)
L4_N = 80               # Level 4 sample size (half with f-in-core lanthanides, half without rare earths)
ML_MODEL = "mace"       # "mace" or "chgnet"
HHI_OVERRIDE_CSV = None # optional CSV: element,hhi_p[,hhi_r] with newer supply data
USGS_LOCAL_FILE = None  # only if the automatic USGS download fails: path to the uploaded world data file
N_BOOT = 300            # bootstrap resamples
N_NULL = 100            # null-calibration repeats (use 200 for the final paper run)
SEED = 0
USE_CHECKPOINT = True  # reuse the saved main table on Drive (skips section 4 on later runs)

import os, json, time, warnings, importlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
import crm_audit as ca, crm_v2 as v2, crm_v3 as v3
importlib.reload(ca); importlib.reload(v2); importlib.reload(v3)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE = "/content/drive/MyDrive/crm_audit_cache"
else:
    CACHE = "cache"
RESULTS = "results"
os.makedirs(CACHE, exist_ok=True); os.makedirs(RESULTS, exist_ok=True)

API_KEY = None
if not DEMO_MODE:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("MP_API_KEY")
    except Exception:
        from getpass import getpass
        API_KEY = getpass("Materials Project API key: ")

def save(df, name):
    df.to_csv(f"{RESULTS}/{name}.csv", index=False)
    return df

## 3. Load all data
Materials Project (all materials + two kinds of energies), GNoME summary, GNoME r²SCAN file, DeepMind's external hull file (OQMD, WBM, ...), Energy-GNoME shortlists. First real run: ~10–20 min. Later: ~1 min.

In [ ]:
t0 = time.time()
r2raw = xraw = eg = usgs_raw = None
if DEMO_MODE:
    mp_c, graw, r2raw, xraw, eg, thermo = v2.make_demo_sources(seed=SEED)
    usgs_raw = v3.make_demo_usgs(seed=SEED)
else:
    mp_c = v2.load_mp_compact(API_KEY, CACHE)
    print("MP materials:", len(mp_c))
    thermo = {t: ca.fetch_thermo(API_KEY, t, f"{CACHE}/thermo_{t.replace('+', 'p')}.parquet", chunk_size=1000)
              for t in (ca.THERMO_GGA, ca.THERMO_R2)}
    graw = v2.load_gnome_raw(CACHE)
    for name, loader in [("r2raw", v2.load_gnome_r2scan), ("xraw", v2.load_external), ("eg", v2.load_energy_gnome)]:
        try:
            globals()[name] = loader(CACHE)
            print(name, "rows:", len(globals()[name]))
        except Exception as e:
            print(f"!! could not load {name}: {e}")
    try:
        usgs_raw = v3.fetch_usgs_world(CACHE, local_file=USGS_LOCAL_FILE)
        print("USGS rows:", len(usgs_raw))
    except Exception as e:
        print(f"!! USGS world data not loaded ({e}). Download the 'World' CSV from "
              "https://doi.org/10.5066/P13XCP3R, upload it to Colab and set USGS_LOCAL_FILE.")
print(f"loaded in {time.time() - t0:.0f}s")

## 4. Build one table
All groups in one table, with every supply-risk measure per compound (~5 min on real data).
DeepMind's external file has no source column. Its IDs are either Materials Project IDs (`EXT_MP`) or coded IDs
from other databases (`EXT_CODED`), so entries are grouped by ID type. The finished table is saved to Drive.

In [ ]:
t0 = time.time()
CKPT = f"{CACHE}/checkpoint/master.parquet"
if USE_CHECKPOINT and not DEMO_MODE and os.path.exists(CKPT):
    master = pd.read_parquet(CKPT)
    print("loaded saved table:", CKPT)
else:
    parts = [v2.mp_to_master(mp_c), v2.gnome_to_master(graw)]
    if xraw is not None:
        parts.append(v2.external_to_master(xraw))
    master = v2.add_metrics(pd.concat(parts, ignore_index=True), hhi=v2.load_hhi(HHI_OVERRIDE_CSV))
    ext = master["material_id"].str.startswith("ext-")
    ids = master.loc[ext, "material_id"].str[4:]
    master.loc[ext, "group"] = np.select(
        [ids.str.lower().str.startswith("mp-"), ids.str.fullmatch(r"[0-9a-fA-F]{8,}")],
        ["EXT_MP", "EXT_CODED"], default="EXT_OTHER")
    master["s_n5_class"] = master["nelements"].clip(upper=5).astype(str) + "_" + master["chem_class"]
    if not DEMO_MODE:
        try:
            os.makedirs(os.path.dirname(CKPT), exist_ok=True)
            obj = master.select_dtypes("object").columns
            master.astype({c: str for c in obj}).to_parquet(CKPT, index=False)
            print("saved table to", CKPT)
        except Exception as e:
            print("!! could not save the table:", e)
G = master[master["group"] == "GNoME"].copy()
REFS = ["MP_known", "MP_theoretical"] + [g for g in ["EXT_MP", "EXT_CODED", "EXT_OTHER"] if (master.group == g).any()]
print(f"built in {time.time() - t0:.0f}s")
print(master["group"].value_counts())

### 4b. Country data (USGS 2025)
Per element: which countries produce it, the new HHI, the top country and China's share. Elements marked `proxy` borrow a related commodity's pattern (e.g. each rare earth uses the rare-earth total).

In [ ]:
summ = shares = None
if usgs_raw is not None:
    long = v3.parse_usgs_world(usgs_raw)
    print("production column used:", long.attrs.get("year_column"))
    summ, shares = v3.element_country_shares(long)
    save(summ, "country_element_table")
    print("elements with country data:", int(summ.hhi_new.notna().sum()), "of", len(summ))
    print("missing:", summ.loc[summ.hhi_new.isna(), "element"].tolist())
    print(summ.dropna(subset=["hhi_new"]).sort_values("top_share", ascending=False)
          [["element", "source", "top_country", "top_share", "china_share", "hhi_new", "proxy"]].head(20).round(3).to_string(index=False))
    master = v3.add_country_metrics(master, summ)
    G = master[master["group"] == "GNoME"].copy()
    if DEMO_MODE:
        eg = v3.add_demo_properties(eg, master, seed=SEED)
else:
    print("No USGS data: country sections will be skipped.")

## 5. Data checks
Read these before any result.

In [ ]:
mp_formulas_known = set(master.loc[master.group == "MP_known", "formula"])
mp_formulas_all = set(master.loc[master.group.isin(["MP_known", "MP_theoretical"]), "formula"])
print("GNoME formulas also in MP (known):", int(G["formula"].isin(mp_formulas_known).sum()),
      "| in MP (any):", int(G["formula"].isin(mp_formulas_all).sum()))
print("  (DeepMind removed overlaps with MP/OQMD snapshots of June 2024, so ~0 is expected.)")
print("GNoME rows:", len(G), "| unique formulas:", G["formula"].nunique())
print("GNoME 'Is Train' known for:", int(G["is_train"].notna().sum()), "| margin column known for:",
      int(G["e_decomp_rel"].notna().sum()))
print("Crystal system known (MP):", int((master.loc[master.group.str.startswith("MP"), "crystal_system"] != "unknown").sum()))
if r2raw is not None: print("r2SCAN file columns:", list(r2raw.columns))
if xraw is not None: print("External file columns:", list(xraw.columns))
if eg is not None: print("Energy-GNoME rows per application:", eg["application"].value_counts().to_dict())
print("\nChemistry type share per group:")
print(pd.crosstab(master["group"], master["chem_class"], normalize="index").round(3))

# Level 1 — is the main result solid?
## 6. Test A, all variants
Each GNoME value is compared with a reference group re-weighted to the same mix of number of elements (`s_n`), plus chemistry type (`s_n_class`), plus crystal system (`s_n_class_xtal`).

Fast confidence intervals: the reference mean is re-weighted to GNoME's mix, and the interval uses the analytic standard error, Var = var_T/n_T + Σ π_s² var_R(s)/n_R(s). This replaces the slow bootstrap.

In [ ]:
import numpy as np, pandas as pd

def fast_compare_metric(df, metric, target="GNoME", ref="MP_known", strata_col="nelements", n_boot=None, seed=None):
    d = df.loc[df["group"].isin([target, ref]), ["group", strata_col, metric]].dropna(subset=[metric])
    T, R = d[d["group"] == target], d[d["group"] == ref]
    common = set(T[strata_col]) & set(R[strata_col])
    if not common:
        raise ZeroDivisionError("no common strata")
    T = T[T[strata_col].isin(common)]
    tv = T[metric].astype(float).values
    pi = T[strata_col].value_counts(normalize=True)
    Rc = R[R[strata_col].isin(common)].assign(_v=lambda x: x[metric].astype(float))
    g = Rc.groupby(strata_col)["_v"].agg(["mean", "var", "size"]).loc[pi.index]
    ref_m = float((pi * g["mean"]).sum())
    var_ref = float((pi ** 2 * g["var"].fillna(0) / g["size"]).sum())
    var_t = tv.var(ddof=1) / len(tv) if len(tv) > 1 else 0.0
    diff, se = tv.mean() - ref_m, np.sqrt(var_t + var_ref)
    return {"metric": metric, "target": target, "ref": ref, "mean_target": tv.mean(),
            "mean_ref_matched": ref_m, "mean_ref_raw": R[metric].astype(float).mean(),
            "diff": diff, "ci_low": diff - 1.96 * se, "ci_high": diff + 1.96 * se,
            "n_target": len(tv), "n_ref": len(R)}

def fast_compare_grid(df, metrics, refs, strata_list, target="GNoME", n_boot=None, seed=0, label=""):
    rows = []
    for ref in refs:
        sub = df[df["group"].isin([target, ref])]
        if (sub["group"] == ref).sum() == 0:
            continue
        for st in strata_list:
            s2 = sub[~sub[st].astype(str).str.contains("unknown|nan|None")] if st == "s_n_class_xtal" else sub
            for m in metrics:
                try:
                    r = ca.compare_metric(s2, m, target, ref, strata_col=st)
                except (ZeroDivisionError, ValueError, IndexError, KeyError):
                    continue
                rows.append(dict(check=label, ref=ref, matched_on=st, metric=m, target=r["mean_target"],
                                 ref_matched=r["mean_ref_matched"], ref_raw=r["mean_ref_raw"], diff=r["diff"],
                                 ci_low=r["ci_low"], ci_high=r["ci_high"], n_target=r["n_target"], n_ref=r["n_ref"]))
    return pd.DataFrame(rows)

ca.compare_metric = fast_compare_metric
v2.compare_grid = fast_compare_grid
print("fast version on")

In [ ]:
t0 = time.time()
cols = ["group", "s_n", "s_n_class", "s_n_class_xtal", "s_n5_class", "formula", "e_decomp_rel", "has_radioactive"] + v2.HEADLINE_METRICS
slim = master[[c for c in dict.fromkeys(cols) if c in master.columns]].copy()
parts = []
for r in REFS:
    st = ["s_n", "s_n_class", "s_n_class_xtal"] if r.startswith("MP_") else ["s_n", "s_n_class"]
    parts.append(v2.compare_grid(slim, v2.HEADLINE_METRICS, [r], st, label="main"))
    print(r, f"done ({time.time() - t0:.0f}s)", flush=True)
extra = []
for label, d in [("duplicates removed", v2.dedupe_by_formula(slim, "e_decomp_rel")),
                 ("radioactive removed", slim[~slim.has_radioactive])]:
    extra.append(v2.compare_grid(d, ["crm_eu_atom", "crm_us_mass", "has_ree", "has_f_in_core"], ["MP_known"], ["s_n_class"], label=label))
gridA = save(pd.concat(parts + extra, ignore_index=True), "L1_testA_grid")
show = gridA[(gridA.ref == "MP_known") & (gridA.matched_on == "s_n_class")]
print(show[["check", "metric", "target", "ref_matched", "diff", "ci_low", "ci_high"]].round(3).to_string(index=False))

In [ ]:
key = ["crm_eu_atom", "crm_us_atom", "has_ree", "has_f_in_core", "has_pgm"]
names = {"crm_eu_atom": "EU critical\n(atom share)", "crm_us_atom": "US critical\n(atom share)",
         "has_ree": "contains\nrare earth", "has_f_in_core": "contains f-in-core\nlanthanide",
         "has_pgm": "contains\nPt-group metal"}
s = gridA[(gridA.check == "main") & (gridA.matched_on == "s_n_class") & gridA.metric.isin(key)]
refs_here = [r for r in REFS if r in s.ref.unique()]
colors = {"MP_known": "#4c72b0", "MP_theoretical": "#8fa8d0", "EXT_MP": "#a1c99a", "EXT_CODED": "#55a868", "EXT_OTHER": "#8c8c8c"}
fig, ax = plt.subplots(figsize=(9, 4))
w = 0.8 / (len(refs_here) + 1); x = np.arange(len(key))
ax.bar(x, [s[s.metric == m].target.iloc[0] for m in key], w, label="GNoME", color="tab:red")
for i, r in enumerate(refs_here):
    vals = [s[(s.metric == m) & (s.ref == r)].ref_matched.iloc[0] for m in key]
    ax.bar(x + (i + 1) * w, vals, w, label=r + " (matched)", color=colors.get(r, "#999999"))
ax.set_xticks(x + w * len(refs_here) / 2, [names[m] for m in key], fontsize=8)
ax.set_ylabel("share"); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{RESULTS}/fig1_headline.png", dpi=200); plt.show()
print("Fig. 1. GNoME vs reference groups, matched on number of elements and chemistry type.")
print(s[["ref", "metric", "target", "ref_matched", "diff", "ci_low", "ci_high"]].round(3).to_string(index=False))

### 6b. Fair comparison only where known materials exist (common support)
Compounds with 5 or more elements are grouped as "5+". Only element-count × chemistry groups with at least 100
reference compounds are compared. The share of GNoME covered is reported (72.6% for known materials in the paper run).

In [ ]:
metrics = ["crm_eu_atom", "crm_us_atom", "has_ree", "has_f_in_core", "has_pgm", "hhi_p"]
REFS3 = [r for r in ["MP_known", "MP_theoretical", "EXT_CODED"] if (master.group == r).any()]
slim5 = master[["group", "s_n5_class"] + metrics]
ct = pd.crosstab(slim5.s_n5_class, slim5.group)
print(ct[["GNoME"] + REFS3])
r5 = v2.compare_grid(slim5, metrics, REFS3, ["s_n5_class"], label="5+ elements grouped")
save(r5, "L1_testA_capped")
rows = []
for ref in REFS3:
    ok = ct.index[ct[ref] >= 100]
    r = v2.compare_grid(slim5[slim5.s_n5_class.isin(ok)], metrics, [ref], ["s_n5_class"], label="common support (ref >= 100)")
    r["gnome_share_covered"] = ct.loc[ok, "GNoME"].sum() / ct["GNoME"].sum()
    rows.append(r)
cs = save(pd.concat(rows, ignore_index=True), "L1_testA_common_support")
print(cs[["ref", "gnome_share_covered", "metric", "target", "ref_matched", "diff", "ci_low", "ci_high"]].round(3).to_string(index=False))

## 7. Which elements are over-produced

In [ ]:
enr = save(ca.element_enrichment(master, "GNoME", "MP_known", n_boot=min(N_BOOT, 200), seed=SEED), "L1_element_enrichment")
show = pd.concat([enr.head(15), enr.tail(15)])
col = ["tab:red" if e in ca.F_IN_CORE else ("tab:orange" if e in ca.REE else
       ("tab:purple" if e in {"Ru","Rh","Pd","Os","Ir","Pt"} else "tab:gray")) for e in show.element]
fig, ax = plt.subplots(figsize=(6, 7))
ax.barh(show.element, show.log2_enrichment, xerr=[show.log2_enrichment - show.ci_low, show.ci_high - show.log2_enrichment], color=col)
ax.axvline(0, color="k", lw=.8); ax.invert_yaxis(); ax.set_xlabel("log2 enrichment (GNoME vs known, matched)")
plt.tight_layout(); plt.savefig(f"{RESULTS}/fig2_enrichment.png", dpi=200); plt.show()
print("Fig. 2. Most and least over-produced elements. Red: f-in-core lanthanides; orange: other rare earths; purple: Pt-group.")
print("Family test (f-in-core):", ca.family_test(enr))

## 8. Is this just what a random element choice would give?
If each compound picked its elements at random from the periodic table, a big family like the 17 rare earths would appear often by chance. `ratio` > 1 means *more* than random choice; < 1 means less.

In [ ]:
fams = [("rare earths", ca.REE), ("f-in-core lanthanides", ca.F_IN_CORE),
        ("Pt-group metals", ["Ru", "Rh", "Pd", "Os", "Ir", "Pt"]), ("EU critical", v2.CRM_EU)]
uni = save(pd.concat([v2.uniform_expectation(master, ["GNoME"] + REFS, f, lab) for lab, f in fams]), "L1_uniform_expectation")
print(uni.round(3).to_string(index=False))

# Level 2 — why does it happen?
## 9. How much of the gap comes from chemistry mix?
`mix` = part explained by GNoME making more metal-only compounds (and different element counts). `within` = part that remains inside the same kind of chemistry.

In [ ]:
dec = []
for m in ["crm_eu_atom", "crm_us_atom", "has_ree", "has_f_in_core", "hhi_p"]:
    r = v2.decompose_gap(master, m, "GNoME", "MP_known", "s_n_class", n_boot=min(N_BOOT, 200), seed=SEED)
    dec.append({k: (v if not isinstance(v, tuple) else f"({v[0]:.3f}, {v[1]:.3f})") for k, v in r.items()})
dec = save(pd.DataFrame(dec), "L2_gap_decomposition")
print(dec.to_string(index=False))
d = dec[dec.metric != "hhi_p"]
fig, ax = plt.subplots(figsize=(6, 3.5))
lab = d.metric.map({"crm_eu_atom": "EU critical\nshare", "crm_us_atom": "US critical\nshare",
                     "has_ree": "contains\nrare earth", "has_f_in_core": "contains f-in-core\nlanthanide"})
ax.bar(lab, d["mix"].astype(float), label="chemistry mix", color="tab:blue")
ax.bar(lab, d["within"].astype(float), bottom=d["mix"].astype(float), label="within same chemistry", color="tab:red")
ax.set_ylabel("gap (GNoME − known)"); ax.legend(fontsize=8); plt.xticks(fontsize=8)
plt.tight_layout(); plt.savefig(f"{RESULTS}/fig3_decomposition.png", dpi=200); plt.show()
print("Fig. 3. The gap split into chemistry-mix and within-chemistry parts.")

In [ ]:
# Same split, on common support only (the version quoted in the paper)
ok = ct.index[ct["MP_known"] >= 100]
dec5 = []
for m in metrics:
    r = v2.decompose_gap(slim5[slim5.s_n5_class.isin(ok)], m, "GNoME", "MP_known", "s_n5_class", n_boot=200, seed=SEED)
    dec5.append({k: (v if not isinstance(v, tuple) else f"({v[0]:.3f}, {v[1]:.3f})") for k, v in r.items()})
dec5 = save(pd.DataFrame(dec5), "L2_gap_decomposition_common_support")
print(dec5[["metric", "gap", "mix", "within", "mix_share", "mix_share_ci"]].to_string(index=False))

## 10. Is the bias inherited from computed data?
Computed (theoretical) Materials Project entries are GNoME's starting point. If GNoME's element bias follows the bias already present in computed data, the slope tells us whether GNoME *amplifies* it (slope > 1).

In [ ]:
amp, ampj = v2.amplification(master, n_boot=50, seed=SEED)
save(ampj, "L2_amplification"); print(amp)
fig, ax = plt.subplots(figsize=(5.5, 5))
c = ["tab:red" if e in ca.F_IN_CORE else ("tab:orange" if e in ca.REE else "tab:gray") for e in ampj.element]
ax.scatter(ampj.log2_enrichment_mp_theo, ampj.log2_enrichment_gnome, c=c, s=16)
for _, r in ampj.iterrows(): ax.annotate(r.element, (r.log2_enrichment_mp_theo, r.log2_enrichment_gnome), fontsize=6)
lim = [min(ampj.iloc[:, 1:].min()), max(ampj.iloc[:, 1:].max())]; ax.plot(lim, lim, "k--", lw=.7)
ax.set_xlabel("bias in computed MP entries (log2 vs known)"); ax.set_ylabel("bias in GNoME (log2 vs known)")
plt.tight_layout(); plt.savefig(f"{RESULTS}/fig4_amplification.png", dpi=200); plt.show()
print("Fig. 4. Element bias in GNoME vs bias already in computed MP data. Dashed: equal bias.")

gt = master.copy()
gt.loc[(gt.group == "GNoME") & (gt.is_train.astype(str).str.lower() == "true"), "group"] = "GNoME_train"
rows = []
for m in ["crm_eu_atom", "has_ree", "has_f_in_core"]:
    if (gt.group == "GNoME_train").sum():
        r = ca.compare_metric(gt.assign(**{m: gt[m].astype(float)}), m, "GNoME_train", "GNoME", "s_n_class", n_boot=200, seed=SEED)
        rows.append(dict(metric=m, in_training=r["mean_target"], not_in_training_matched=r["mean_ref_matched"], diff=r["diff"], ci_low=r["ci_low"], ci_high=r["ci_high"]))
print("\nGNoME entries used in model training vs the rest:")
print(save(pd.DataFrame(rows), "L2_is_train").round(3).to_string(index=False))

## 11. Near-copies?
Replace any rare earth with 'Ln' and count how many compounds share each template.

In [ ]:
print(save(v2.ln_templates(master), "L2_templates").round(3).to_string(index=False))

## 12. How safely stable are the GNoME compounds?
DeepMind gives each compound's distance below the hull of all other entries (`margin`). A thin margin (< 10 or < 25 meV/atom) is within typical calculation error, so the compound is fragile. Odds ratio > 1: compounds with the flag are more often fragile (pooled over number of elements and chemistry type).

In [ ]:
cols = ["e_decomp", "e_decomp_all", "e_decomp_rel", "e_decomp_mp", "e_decomp_mp_oqmd"]
print("values available per column:", {c: int(G[c].notna().sum()) for c in cols if c in G.columns})

margin_col = next(c for c in ["e_decomp_rel", "e_decomp_mp", "e_decomp_mp_oqmd", "e_decomp"]
                  if c in G.columns and G[c].notna().sum() > 1000)
print("using:", margin_col)
mt, gm = v2.margin_table(G, margin_col=margin_col)
print(save(mt, "L2_stability_margins").round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3.5))
bins = np.linspace(0, 300, 61)
for val, lab, colr in [(True, "with f-in-core lanthanide", "tab:red"), (False, "without", "tab:gray")]:
    ax.hist(gm.loc[gm.has_f_in_core == val, "margin_meV"].clip(lower=0, upper=300), bins, density=True, alpha=.55, label=lab, color=colr)
ax.axvline(25, color="k", ls="--", lw=.8); ax.set_xlabel(f"stability margin (meV/atom), from {margin_col}"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{RESULTS}/fig5_margins.png", dpi=200); plt.show()
print("Fig. 5. Stability margins of GNoME compounds. Dashed: 25 meV/atom.")

## 13. Do GNoME compounds stay stable with r²SCAN?
Uses DeepMind's own r²SCAN file. **Caveat (verified):** Materials Project's GGA and r²SCAN settings both freeze the 4f electrons of Pr–Lu (pymatgen `MPRelaxSet.yaml`, `MPSCANRelaxSet.yaml`), so this check cannot see 4f errors. It tests sensitivity to the functional only.

In [ ]:
r2tab = None
if r2raw is not None:
    info, r2tab, r2j = v2.r2scan_check(r2raw, G)
    print({k: v for k, v in info.items() if k != "columns"})
    if r2tab is None:
        print("!! No stability column recognised. Columns are:", info["columns"], "-> send these to adapt the test.")
    else:
        print(save(r2tab, "L2_r2scan_gnome").round(3).to_string(index=False))
# Same check inside Materials Project (GGA hull -> r2SCAN hull), for comparison
surv = ca.build_survival_table(master[master.group.isin(["MP_known", "MP_theoretical"])],
                               thermo[ca.THERMO_GGA][["material_id", "e_hull"]], thermo[ca.THERMO_R2][["material_id", "e_hull"]], thr=0.0)
mp_or = pd.DataFrame([dict(group=g, flag=f, **{k: v for k, v in ca.mantel_haenszel_or(surv, f, g, "s_n_class").items() if k in ("or_mh", "ci_low", "ci_high")})
                      for f in ["has_f_in_core", "has_ree", "has_u_element"] for g in ["MP_known", "MP_theoretical"]])
print("\nInside Materials Project (odds of losing stability under r2SCAN):")
print(save(mp_or, "L2_r2scan_mp").round(3).to_string(index=False))

## 14. Test B — link to energy sensitivity (pre-registered H2)

In [ ]:
sens = ca.element_functional_sensitivity(master, thermo[ca.THERMO_GGA], thermo[ca.THERMO_R2], groups=("MP_known",), value="ef")
counts = master[master.group == "MP_known"]["elements"].str.split(",").explode().value_counts().to_dict()
rows = []
for lab, d in [("all compounds", master), ("metal-only compounds", master[master.chem_class == "intermetallic"])]:
    e = enr if lab == "all compounds" else ca.element_enrichment(d, n_boot=30, seed=SEED)
    s_ = sens if lab == "all compounds" else ca.element_functional_sensitivity(d, thermo[ca.THERMO_GGA], thermo[ca.THERMO_R2], groups=("MP_known",), value="ef")
    lt, _ = ca.link_test(e, s_, counts, n_perm=5000, seed=SEED)
    rows.append(dict(subset=lab, **{k: float(v) for k, v in lt.items()}))
print(save(pd.DataFrame(rows), "L2_testB").round(4).to_string(index=False))

# Level 3 — does it matter in practice?
## 15. Energy-GNoME application shortlists
GNoME candidates already shortlisted for batteries (cathodes), solar (perovskites) and thermoelectrics.

In [ ]:
egt = None
if eg is not None:
    egt, egj = v2.energy_gnome_table(eg, master, n_boot=200, seed=SEED)
    save(egt, "L3_energy_gnome")
    print(egt[egt.get("metric", pd.Series(dtype=str)).isin(["crm_eu_atom", "has_ree", "has_f_in_core"])]
          [["application", "subset", "n_matched", "metric", "shortlist", "all_gnome", "known_matched", "ci_low", "ci_high"]].round(3).to_string(index=False))
    s = egt[(egt.metric == "crm_eu_atom") & (egt.subset == "all")]
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.bar(np.arange(len(s)) - .2, s.shortlist, .4, label="shortlist", color="tab:red")
    ax.bar(np.arange(len(s)) + .2, s.known_matched, .4, label="known (matched)", color="tab:gray")
    ax.set_xticks(np.arange(len(s)), s.application); ax.set_ylabel("EU critical atom share"); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(f"{RESULTS}/fig6_applications.png", dpi=200); plt.show()
    print("Fig. 6. Critical share in application shortlists vs known materials.")
else:
    print("Energy-GNoME data not loaded.")

## 16. Country exposure
How much of each group's supply depends on single countries (USGS 2025 production data). `china_frac` = atom-weighted China share of world production; `conc_frac` = share of atoms from elements where one country makes at least half the world supply; `top_share_max` = the most concentrated element in the compound.

In [ ]:
gridC = None
if summ is not None:
    gridC = save(v2.compare_grid(master, ["hhi_new", "china_frac", "conc_frac", "top_share_max"], REFS,
                                 ["s_n", "s_n_class"], n_boot=N_BOOT, seed=SEED, label="country"), "L3_country_grid")
    print(gridC[gridC.ref == "MP_known"][["matched_on", "metric", "target", "ref_matched", "diff", "ci_low", "ci_high"]].round(3).to_string(index=False))
    prof = v3.country_profile(master, shares, ["GNoME"] + REFS)
    prof.to_csv(f"{RESULTS}/L3_country_profile.csv")
    print("\nShare of supply by country (reference groups re-weighted to GNoME's chemistry mix):")
    print(prof.round(3))
    fig, ax = plt.subplots(figsize=(7, 4))
    top = prof.index[:8]; x = np.arange(len(top)); w = 0.8 / len(prof.columns)
    for i, gname in enumerate(prof.columns):
        ax.bar(x + i * w, prof.loc[top, gname], w, label=gname,
               color="tab:red" if gname == "GNoME" else ["#4c72b0", "#8fa8d0", "#55a868", "#a1c99a", "#8c8c8c"][(i - 1) % 5])
    ax.set_xticks(x + w * (len(prof.columns) - 1) / 2, top, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("share of supply"); ax.legend(fontsize=7)
    plt.tight_layout(); plt.savefig(f"{RESULTS}/fig7_countries.png", dpi=200); plt.show()
    print("Fig. 7. Which countries supply the elements in each group (USGS 2025, matched mix).")

In [ ]:
# Country exposure on common support
cc = None
if summ is not None:
    cslim = master[["group", "s_n5_class", "hhi_new", "china_frac", "conc_frac", "top_share_max"]]
    ctc = pd.crosstab(cslim.s_n5_class, cslim.group)
    crow = []
    for ref in REFS3:
        ok = ctc.index[ctc[ref] >= 100]
        r = v2.compare_grid(cslim[cslim.s_n5_class.isin(ok)], ["hhi_new", "china_frac", "conc_frac", "top_share_max"],
                            [ref], ["s_n5_class"], label="country, common support")
        r["gnome_share_covered"] = ctc.loc[ok, "GNoME"].sum() / ctc["GNoME"].sum()
        crow.append(r)
    cc = save(pd.concat(crow, ignore_index=True), "L3_country_common_support")
    print("Country exposure, common support:")
    print(cc[["ref", "gnome_share_covered", "metric", "target", "ref_matched", "diff", "ci_low", "ci_high"]].round(3).to_string(index=False))

## 17. Lower-risk alternatives among application shortlists
For each shortlist (battery cathodes per working ion, perovskite solar absorbers, thermoelectrics per temperature): how many high-risk candidates (EU critical share ≥ 0.5) have an alternative with near-equal predicted performance (within 5% or 10%; band gap within 0.05 or 0.1 eV of 1.34 eV) and risk lower by at least 0.2? And how much performance do we lose if we keep only low-risk candidates? Only candidates with AI-expert confidence ≥ 0.5.

**First design (kept for the record, not used in the paper).** It counted near-identical candidates as alternatives and gave 100% / zero loss. Its figure (Fig. 8, the trade-off plot) is still valid.

In [ ]:
altT = None
if eg is not None and len(eg):
    egj = v3.eg_join(eg, master)
    tabs = []
    for risk in ["crm_eu_atom"] + (["conc_frac"] if "conc_frac" in egj.columns else []):
        for tol in (0.05, 0.10):
            a, per = v3.alternatives(egj, risk=risk, rel_tol=tol, abs_tol=tol)
            tabs.append(a.assign(risk_measure=risk, tolerance=tol))
            if risk == "crm_eu_atom" and tol == 0.10:
                per_main = per
    altT = save(pd.concat(tabs, ignore_index=True), "L3_alternatives")
    print(altT[["risk_measure", "tolerance", "application", "subset", "n", "high_risk_share",
                "high_risk_with_alternative", "pareto_high_risk_share", "top10_loss", "loss_unit"]].round(3).to_string(index=False))
    apps = [a for a in ["cathode", "perovskite", "thermoelectric"] if len(per_main) and a in per_main.application.unique()]
    if not apps: print("No shortlist had enough candidates with performance data.")
    fig, axs = plt.subplots(1, max(len(apps), 1), figsize=(4 * max(len(apps), 1), 3.6))
    for ax, app in zip(np.atleast_1d(axs), apps):
        d = per_main[per_main.application == app]
        sub = d.subset.value_counts().index[0]; d = d[d.subset == sub]
        ax.scatter(d.crm_eu_atom, d.perf, s=8, c=np.where(d.high_risk, "tab:red", "tab:gray"), alpha=.6)
        f = d[d.pareto].sort_values("crm_eu_atom")
        ax.plot(f.crm_eu_atom, f.perf, "k-", lw=1)
        ax.set_title(f"{app} ({sub})", fontsize=9); ax.set_xlabel("EU critical atom share")
        ax.set_ylabel(v3.APP_PERF[app][0] if v3.APP_PERF[app][1] == "higher" else "−|band gap − 1.34| (eV)", fontsize=8)
    plt.tight_layout(); plt.savefig(f"{RESULTS}/fig8_alternatives.png", dpi=200); plt.show()
    print("Fig. 8. Predicted performance vs supply risk in shortlists. Red: high risk. Line: best trade-off (Pareto front).")
else:
    print("Energy-GNoME data not loaded.")

**Final design (used in the paper).** The top 1% of predicted performances are removed as likely outliers.
An alternative must have similar predicted performance (within 5% or 10%) and a critical share lower by at least 0.2.

In [ ]:
from scipy import stats as _st

def alternatives_v2(egj, risk="crm_eu_atom", high_risk=0.5, rel_tol=0.10, abs_tol=0.10,
                    min_risk_drop=0.2, conf_min=0.5, trim_top=0.01):
    rows = []
    for (app, sub), d in egj.groupby(["application", "subset"]):
        if app not in v3.APP_PERF:
            continue
        d = d.copy()
        d["perf"] = v3._performance(d, app)
        if v3.CONF_COL in d.columns:
            d = d[pd.to_numeric(d[v3.CONF_COL], errors="coerce") >= conf_min]
        d = d.dropna(subset=["perf", risk]).drop_duplicates("gnome_id")
        if len(d) < 30:
            continue
        cut = d["perf"].quantile(1 - trim_top)
        n_trim = int((d["perf"] > cut).sum())
        d = d[d["perf"] <= cut]
        mode = v3.APP_PERF[app][1]
        p, r = d["perf"].values.astype(float), d[risk].values.astype(float)
        tol = rel_tol * np.abs(p) if mode == "higher" else np.full(len(p), abs_tol)
        hr = r >= high_risk
        order = np.argsort(p); ps, rs = p[order], r[order]
        n_alt = np.zeros(len(p), int)
        for i in np.where(hr)[0]:
            lo, hi = np.searchsorted(ps, p[i] - tol[i]), np.searchsorted(ps, p[i] + tol[i], side="right")
            n_alt[i] = int((rs[lo:hi] <= r[i] - min_risk_drop).sum())
        lowp, highp = p[~hr], p[hr]
        mw = _st.mannwhitneyu(highp, lowp, alternative="two-sided") if len(lowp) > 5 and len(highp) > 5 else None
        rows.append(dict(application=app, subset=sub, n=len(d), trimmed_outliers=n_trim,
                         high_risk_share=hr.mean(),
                         share_with_1plus_alt=(n_alt[hr] >= 1).mean() if hr.any() else np.nan,
                         share_with_3plus_alt=(n_alt[hr] >= 3).mean() if hr.any() else np.nan,
                         median_alts=float(np.median(n_alt[hr])) if hr.any() else np.nan,
                         median_perf_high_risk=np.median(highp) if len(highp) else np.nan,
                         median_perf_low_risk=np.median(lowp) if len(lowp) else np.nan,
                         p90_perf_high_risk=np.quantile(highp, 0.9) if len(highp) else np.nan,
                         p90_perf_low_risk=np.quantile(lowp, 0.9) if len(lowp) else np.nan,
                         mannwhitney_p=mw.pvalue if mw else np.nan,
                         perf_metric=v3.APP_PERF[app][0] + (" (-|gap-1.34| eV)" if mode == "target" else "")))
    return pd.DataFrame(rows)

altF = None
if eg is not None and len(eg):
    egj = v3.eg_join(eg, master)
    altF = pd.concat([alternatives_v2(egj, rel_tol=t, abs_tol=t).assign(tolerance=t) for t in (0.05, 0.10)], ignore_index=True)
    save(altF, "L3_alternatives_v2")
    print("Lower-risk alternatives (top 1% predictions removed; alternatives = similar performance, risk lower by >= 0.2):")
    print(altF[altF.tolerance == 0.10][["application", "subset", "n", "trimmed_outliers", "high_risk_share",
          "share_with_1plus_alt", "share_with_3plus_alt", "median_alts", "median_perf_high_risk",
          "median_perf_low_risk", "p90_perf_high_risk", "p90_perf_low_risk", "mannwhitney_p"]].round(3).to_string(index=False))

## 18. Other large computed databases (not GNoME)
Rows with `ref` = `EXT_*` in section 6 compare GNoME with entries from DeepMind's external hull file (e.g. OQMD = systematic prototype decoration, WBM = systematic element substitution). If these also show the bias, it comes from *searching for stability*, not from AI itself.

In [ ]:
ext = gridA[(gridA.check == "main") & gridA.ref.str.startswith("EXT_") & (gridA.matched_on == "s_n_class")]
print(ext[["ref", "metric", "target", "ref_matched", "ref_raw", "diff", "ci_low", "ci_high", "n_ref"]].round(3).to_string(index=False)
      if len(ext) else "No external groups loaded.")
print("\nRandom-choice comparison for external groups (section 8):")
print(uni[uni.group.str.startswith("EXT_")].round(3).to_string(index=False))

# Level 4 — own calculations (optional)
## 19. Re-check a sample with a universal ML potential
For each sampled GNoME compound, the compound **and** all Materials Project phases of its chemical system are relaxed with the same ML potential; then we ask how far the compound lies from that hull. Systems mixing GGA and GGA+U (U-element with O/F) are left out. **Caveat:** the ML potential learned from Materials Project data, so it shares the frozen-4f treatment; this tests agreement between independent models, not 4f physics. Needs a GPU runtime (Runtime → Change runtime type → T4 GPU). Progress is saved to Drive, so you can rerun to resume.

In [ ]:
l4 = None
if DEMO_MODE:
    # Tiny toy run with a simple built-in potential, only to test the code path.
    import zipfile
    from pymatgen.core import Structure, Lattice
    fcc = lambda el, a: Structure(Lattice.cubic(a), [el]*4, [[0,0,0],[.5,.5,0],[.5,0,.5],[0,.5,.5]])
    l12 = lambda a_, b_, a: Structure(Lattice.cubic(a), [a_, b_, b_, b_], [[0,0,0],[.5,.5,0],[.5,0,.5],[0,.5,.5]])
    toy = {"t1": l12("Au","Cu",3.75), "t2": l12("Ni","Al",3.6), "t3": l12("Ag","Pd",3.95)}
    with zipfile.ZipFile(f"{CACHE}/toy.zip", "w") as z:
        for k, s in toy.items(): z.writestr(f"by_id/{k}.CIF", s.to(fmt="cif"))
    db = {e: fcc(e, a) for e, a in [("Au",4.08),("Cu",3.61),("Ni",3.52),("Al",4.05),("Ag",4.09),("Pd",3.89)]}
    fetch = lambda cs: [(f"mp-{k}", s) for k, s in db.items() if k in cs.split("-")]
    sample = pd.DataFrame({"gnome_id": list(toy), "chemsys": ["Au-Cu","Al-Ni","Ag-Pd"],
                           "elements": ["Au,Cu","Al,Ni","Ag,Pd"], "has_f_in_core": [True, False, True]})
    l4 = v2.run_level4(sample, v2.gnome_structure_reader(local_zip=f"{CACHE}/toy.zip"), fetch,
                       v2.make_calculator("emt"), f"{CACHE}/toy_l4.parquet", log_every=1)
elif RUN_LEVEL4:
    pkg = "mace-torch" if ML_MODEL == "mace" else "chgnet"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
    sample = v2.pick_level4_sample(master, L4_N, seed=SEED)
    print("sample:", len(sample), sample.has_f_in_core.value_counts().to_dict(), sample.chem_class.value_counts().to_dict())
    reader = v2.gnome_structure_reader()
    print("GNoME CIFs available:", reader.n_files)
    l4 = v2.run_level4(sample, reader, v2.mp_structure_fetcher(API_KEY), v2.make_calculator(ML_MODEL),
                       f"{CACHE}/level4_{ML_MODEL}_energies.parquet")
else:
    print("Level 4 skipped (set RUN_LEVEL4 = True on a GPU runtime).")
if l4 is not None:
    save(l4, "L4_results")
    print(l4["status"].value_counts().to_dict())
    print(save(v2.level4_summary(l4), "L4_summary").round(3).to_string(index=False))
    ok = l4[l4.status == "ok"]
    if "e_decomp_mp" in ok.columns and len(ok):
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.scatter(ok.e_decomp_mp, ok.e_hull_ml, c=np.where(ok.has_f_in_core, "tab:red", "tab:gray"), s=14)
        ax.axhline(0, color="k", lw=.7); ax.set_xlabel("GNoME distance to MP hull (eV/atom)"); ax.set_ylabel(f"{ML_MODEL} distance to hull (eV/atom)")
        plt.tight_layout(); plt.savefig(f"{RESULTS}/fig9_level4.png", dpi=200); plt.show()
        print("Fig. 9. GNoME's own stability vs ML-potential re-check. Red: with f-in-core lanthanide.")

## 20. Null calibration
On fake data with no effect, each test should raise a false alarm about 5% of the time.

In [ ]:
fa = {"A": [], "family": [], "B": [], "C": []}
for k in range(N_NULL):
    d0, g0, r0 = ca.make_synthetic(n_known=3000, n_theo=1500, n_gnome=4000, effect=False, seed=1000 + k)
    d0 = ca.add_compound_metrics(d0)
    a = ca.compare_metric(d0.assign(x=d0.has_f_in_core.astype(float)), "x", n_boot=200, seed=k)
    fa["A"].append(not (a["ci_low"] <= 0 <= a["ci_high"]))
    e0 = ca.element_enrichment(d0, n_boot=10, seed=k)
    fa["family"].append(ca.family_test(e0)["p_one_sided"] < 0.05)
    l0, _ = ca.link_test(e0, ca.element_functional_sensitivity(d0, g0, r0), None, n_perm=500, seed=k)
    fa["B"].append(l0["p_perm"] < 0.05)
    m0 = ca.mantel_haenszel_or(ca.build_survival_table(d0, g0, r0))
    fa["C"].append(not (m0["ci_low"] <= 1 <= m0["ci_high"]))
null = save(pd.DataFrame([{k: np.mean(v) for k, v in fa.items()}]), "null_calibration")
print(null.round(3).to_string(index=False), f"(target ≈ 0.05, from {N_NULL} runs)")

## 21. Extra checks for the limits
A. Add Al, F, C, Si and He back to the critical lists. B. The large metal-only region compared with other computed
databases. C. Is the r²SCAN subset like the rest of GNoME? D. Margins measured against Materials Project + OQMD.
E. How many compounds contain germanium (no USGS world figure).

In [ ]:
# ===== Extra checks for the Limits section =====
ok = ct.index[ct["MP_known"] >= 100]
REFS4 = ["GNoME", "MP_known", "MP_theoretical"] + [g for g in ["EXT_CODED"] if g in REFS]
m6 = ["crm_eu_atom", "crm_eu_amb_atom", "crm_us_atom", "crm_us_amb_atom"]

# A. Limit 6: add Al, F, C, Si, He back to the critical lists
d6 = master.loc[master.s_n5_class.isin(ok), ["group", "s_n5_class"] + m6]
r6 = v2.compare_grid(d6, m6, ["MP_known"], ["s_n5_class"], label="list check")
print("A. Wider lists (_amb = Al, F, C, Si, He counted as critical):")
print(save(r6, "X_list_check")[["metric", "target", "ref_matched", "diff", "ci_low", "ci_high"]].round(3).to_string(index=False))

# B. Limit 2: the big metal-only region, compared with other computed databases
m2 = ["crm_eu_atom", "crm_us_atom", "has_ree", "has_f_in_core", "has_pgm", "hhi_p"]
big = master[master.s_n5_class.isin(["4_intermetallic", "5_intermetallic"]) &
             master.group.isin(REFS4)]
tb = big.groupby(["s_n5_class", "group"])[m2].mean()
tb["n"] = big.groupby(["s_n5_class", "group"]).size()
print("\nB. 4- and 5+-element metal-only compounds:")
print(save(tb.reset_index(), "X_big_region").round(3).to_string(index=False))

# C. Limit 4: are the 117k r2SCAN-checked materials like the rest of GNoME?
ids = set(r2raw[v2.find_col(r2raw, "materialid")].astype(str)) if r2raw is not None else set()
G2 = G.assign(in_r2scan=G["gnome_id"].astype(str).isin(ids))
rep = G2.groupby("in_r2scan")[["has_ree", "has_f_in_core", "has_u_element", "crm_eu_atom", "nelements"]].mean()
rep["n"] = G2.groupby("in_r2scan").size()
print("\nC. r2SCAN subset vs the rest:")
print(save(rep.reset_index(), "X_r2scan_subset").round(3).to_string(index=False))

# D. Limit 5: margins measured against Materials Project + OQMD
mt2, _ = v2.margin_table(G, margin_col="e_decomp_mp_oqmd")
print("\nD. Margins vs MP + OQMD:")
print(save(mt2, "X_margins_mp_oqmd").round(3).to_string(index=False))

# E. Limit 6: how many compounds contain germanium (no USGS country data)
has_ge = master["elements"].str.split(",").apply(lambda e: "Ge" in e)
print("\nE. Share of compounds with Ge:")
print(has_ge.groupby(master["group"]).mean().round(4).to_string())

## 22. Headline numbers and save
One table with the numbers the paper quotes. Then everything is zipped for download.

In [ ]:
def pick(metric, ref="MP_known", st="s_n_class", check="main"):
    r = gridA[(gridA.metric == metric) & (gridA.ref == ref) & (gridA.matched_on == st) & (gridA.check == check)]
    return r.iloc[0] if len(r) else None
H = []
for m in ["crm_eu_atom", "crm_us_atom", "crm_eu_mass", "has_ree", "has_f_in_core", "has_pgm", "hhi_p"]:
    for st in ["s_n", "s_n_class"]:
        r = pick(m, st=st)
        if r is not None:
            H.append(dict(result=f"{m} GNoME vs known (matched on {st})", value=f"{r.target:.3f} vs {r.ref_matched:.3f}",
                          ci=f"diff {r['diff']:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}]"))
for _, r in dec.iterrows():
    H.append(dict(result=f"share of {r.metric} gap from chemistry mix", value=f"{float(r.mix_share):.2f}", ci=r.mix_share_ci))
H.append(dict(result="amplification slope (GNoME bias vs computed-MP bias)", value=f"{amp['theil_slope']:.2f}", ci=str(amp['slope_ci'])))
for _, r in uni[uni.family.isin(["rare earths", "EU critical"])].iterrows():
    H.append(dict(result=f"{r.family}: observed / random-choice expectation, {r.group}", value=f"{r.ratio:.2f}", ci=""))
for _, r in mt[(mt.flag_value == "odds_ratio") & (mt.flag == "has_f_in_core")].iterrows():
    H.append(dict(result=f"odds of thin margin (<{r.threshold_meV:.0f} meV) with f-in-core", value=f"{r.fragile_share:.2f}", ci=f"[{r.ci_low:.2f}, {r.ci_high:.2f}]"))
if r2tab is not None:
    for _, r in r2tab[(r2tab.flag_value == "odds_ratio") & (r2tab.flag == "has_f_in_core")].iterrows():
        H.append(dict(result=f"odds of r2SCAN instability (>{r.threshold_eV} eV) with f-in-core", value=f"{r.unstable_share:.2f}", ci=f"[{r.ci_low:.2f}, {r.ci_high:.2f}]"))
if egt is not None and len(egt):
    for _, r in egt[(egt.metric == "crm_eu_atom") & (egt.subset == "all")].iterrows():
        H.append(dict(result=f"EU critical share, {r.application} shortlist vs known", value=f"{r.shortlist:.3f} vs {r.known_matched:.3f}", ci=f"[{r.ci_low:.3f}, {r.ci_high:.3f}]"))
if gridC is not None:
    for m in ["china_frac", "conc_frac", "hhi_new"]:
        r = gridC[(gridC.metric == m) & (gridC.ref == "MP_known") & (gridC.matched_on == "s_n_class")]
        if len(r):
            r = r.iloc[0]
            H.append(dict(result=f"{m} (USGS 2025) GNoME vs known (matched on s_n_class)", value=f"{r.target:.3f} vs {r.ref_matched:.3f}",
                          ci=f"diff {r['diff']:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}]"))
for _, r in cs[cs.ref == "MP_known"].iterrows():
    H.append(dict(result=f"{r.metric} GNoME vs known, common support ({r.gnome_share_covered:.1%} of GNoME)",
                  value=f"{r.target:.3f} vs {r.ref_matched:.3f}", ci=f"diff {r['diff']:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}]"))
for _, r in dec5.iterrows():
    H.append(dict(result=f"share of {r.metric} gap from chemistry mix (common support)", value=f"{float(r.mix_share):.2f}", ci=r.mix_share_ci))
if cc is not None:
    for _, r in cc[cc.ref == "MP_known"].iterrows():
        H.append(dict(result=f"{r.metric} (USGS 2025) GNoME vs known, common support", value=f"{r.target:.3f} vs {r.ref_matched:.3f}",
                      ci=f"diff {r['diff']:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}]"))
if altF is not None and len(altF):
    for _, r in altF[altF.tolerance == 0.10].iterrows():
        H.append(dict(result=f"{r.application} ({r.subset}): high-risk share / with >=1 lower-risk alternative",
                      value=f"{r.high_risk_share:.2f} / {r.share_with_1plus_alt:.2f}", ci=f"n={r.n}"))
headline = save(pd.DataFrame(H), "HEADLINE")
print(headline.to_string(index=False))

log = dict(time=time.strftime("%Y-%m-%d %H:%M"), demo=DEMO_MODE, seed=SEED, n_boot=N_BOOT, f_in_core=",".join(ca.F_IN_CORE),
           hhi_override=str(HHI_OVERRIDE_CSV), level4=RUN_LEVEL4)
p = f"{RESULTS}/run_log.csv"; pd.DataFrame([log]).to_csv(p, mode="a", header=not os.path.exists(p), index=False)
import shutil
if not DEMO_MODE and IN_COLAB:
    shutil.copytree(RESULTS, f"{CACHE}/results_copy", dirs_exist_ok=True)
shutil.make_archive("crm_audit_results", "zip", RESULTS)
if IN_COLAB:
    from google.colab import files; files.download("crm_audit_results.zip")